# 🏭 Production LLM Systems
## RAG · MCP · Multi-Agent Systems · LLM-Controlled Robots & Arduino

> **Production-grade implementations** — every pattern used in real deployed systems.

---
## 📚 Table of Contents
1. [Production RAG Systems](#rag)
   - Vector stores, chunking, hybrid retrieval, reranking
   - Advanced: HyDE, Multi-Query, RAG-Fusion, Adaptive RAG
2. [MCP — Model Context Protocol](#mcp)
   - Servers, clients, tools, resources, prompts
   - Production MCP with file system, databases, APIs
3. [Multi-Agent Systems](#agents)
   - ReAct (Reasoning + Acting)
   - Plan-and-Execute
   - Reflection & Self-Improving Agents
   - Multi-Agent Orchestration
4. [LLM → Robot / Arduino Control](#robots)
   - LLM as robot brain, command DSL, safety constraints
   - Arduino serial bridge, sensor feedback loop
   - Simulation environment
---


In [ ]:
# Install all dependencies
import subprocess, sys
pkgs = [
    "openai", "anthropic", "faiss-cpu", "chromadb", "sentence-transformers",
    "rank-bm25", "numpy", "pandas", "matplotlib", "seaborn",
    "httpx", "pydantic", "tiktoken", "rich", "tenacity",
    "pyserial",          # Arduino serial comm
    "langchain-core",    # for prompt templates only
]
subprocess.run([sys.executable,"-m","pip","install","--quiet"] + pkgs, check=False)
print("✅ Ready!")

In [ ]:
import os, json, re, time, math, uuid, hashlib, threading, queue
import asyncio, inspect, textwrap, copy, random
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Callable, Union
from dataclasses import dataclass, field
from datetime import datetime
from functools import wraps
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.progress import track
from rich.syntax import Syntax
import warnings; warnings.filterwarnings("ignore")

console = Console()
plt.rcParams["figure.figsize"] = (12, 5)
sns.set_theme(style="darkgrid")

# ── LLM Client (works with OpenAI / Anthropic / local) ─────────────────────────
class LLMClient:
    """
    Unified LLM client.
    Tries Anthropic first, then OpenAI, then falls back to a local mock.
    Set ANTHROPIC_API_KEY or OPENAI_API_KEY env vars for real calls.
    """

    def __init__(self):
        self.backend  = None
        self.model    = None
        self.client   = None
        self._call_log: List[Dict] = []
        self._setup()

    def _setup(self):
        # Try Anthropic
        try:
            import anthropic
            key = os.environ.get("ANTHROPIC_API_KEY","")
            if key:
                self.client  = anthropic.Anthropic(api_key=key)
                self.backend = "anthropic"
                self.model   = "claude-sonnet-4-5"
                console.print("[green]✓ Anthropic backend[/green]"); return
        except ImportError: pass

        # Try OpenAI
        try:
            import openai
            key = os.environ.get("OPENAI_API_KEY","")
            if key:
                self.client  = openai.OpenAI(api_key=key)
                self.backend = "openai"
                self.model   = "gpt-4o-mini"
                console.print("[green]✓ OpenAI backend[/green]"); return
        except ImportError: pass

        # Mock backend (no API key needed)
        self.backend = "mock"
        console.print("[yellow]⚠ No API key — using mock LLM (deterministic responses).[/yellow]")
        console.print("[yellow]  Set ANTHROPIC_API_KEY or OPENAI_API_KEY for real calls.[/yellow]")

    def complete(self, prompt: str, system: str = "", max_tokens: int = 1024,
                 temperature: float = 0.1) -> str:
        t0 = time.time()
        if self.backend == "anthropic":
            msgs = [{"role":"user","content":prompt}]
            resp = self.client.messages.create(
                model=self.model, max_tokens=max_tokens,
                system=system or "You are a helpful AI assistant.",
                messages=msgs)
            result = resp.content[0].text
        elif self.backend == "openai":
            msgs = []
            if system: msgs.append({"role":"system","content":system})
            msgs.append({"role":"user","content":prompt})
            resp = self.client.chat.completions.create(
                model=self.model, messages=msgs,
                max_tokens=max_tokens, temperature=temperature)
            result = resp.choices[0].message.content
        else:
            result = self._mock_complete(prompt, system)

        self._call_log.append({"prompt_len": len(prompt), "result_len": len(result),
                                 "latency_ms": round((time.time()-t0)*1000)})
        return result

    def _mock_complete(self, prompt: str, system: str) -> str:
        """Deterministic mock responses keyed on prompt keywords."""
        p = prompt.lower()

        # ReAct pattern
        if "thought:" in p or "action:" in p or "react" in p:
            return ("Thought: I need to search for information about the query.\n"
                    "Action: search\nAction Input: relevant query\n"
                    "Observation: Found relevant information.\n"
                    "Thought: I now have enough information.\n"
                    "Final Answer: Based on the retrieved information, the answer is 42.")

        # Plan
        if "plan" in p or "step" in p or "subtask" in p:
            return json.dumps({"plan": [
                {"id":1,"task":"Gather information","tool":"search","args":{"query":"topic"}},
                {"id":2,"task":"Analyze data","tool":"analyze","args":{"data":"{{step_1}}"}},
                {"id":3,"task":"Generate report","tool":"write","args":{"content":"{{step_2}}"}}
            ]})

        # Reflection
        if "reflect" in p or "improve" in p or "critique" in p:
            return ("Reflection: The previous response was adequate but could be improved.\n"
                    "Issues: 1) Missing edge cases, 2) No error handling\n"
                    "Improved: Added comprehensive error handling and edge case coverage.")

        # Robot command
        if "robot" in p or "motor" in p or "servo" in p or "arduino" in p or "move" in p:
            return json.dumps({"commands":[
                {"action":"set_motor","params":{"motor":"left","speed":150,"direction":"forward"}},
                {"action":"set_motor","params":{"motor":"right","speed":150,"direction":"forward"}},
                {"action":"delay","params":{"ms":1000}},
                {"action":"stop","params":{}}
            ],"safety_check":"all_clear","estimated_duration_ms":1200})

        # RAG answer
        if "context" in p or "document" in p or "passage" in p:
            return ("Based on the provided context, the answer is:\n\n"
                    "The main concept involves the interaction between components described "
                    "in the retrieved passages. The key points are:\n"
                    "1. The primary mechanism involves the described process.\n"
                    "2. This leads to measurable outcomes.\n"
                    "3. The evidence supports this conclusion.\n\n"
                    "Confidence: High (directly supported by context)\n"
                    "Sources: [0], [1], [2]")

        # Tool call JSON
        if "tool" in p or "function" in p or "json" in p:
            return json.dumps({"tool": "search", "args": {"query": "extracted query from prompt"},
                               "reasoning": "Need to find information"})

        return (f"I understand your request. Based on my analysis, the most appropriate "
                f"response addresses the key aspects of your query about: "
                f"'{prompt[:60].strip()}...' The solution involves careful consideration "
                f"of the context and available information.")

    def stats(self):
        if not self._call_log: return
        avg_lat = np.mean([x["latency_ms"] for x in self._call_log])
        console.print(f"LLM calls: {len(self._call_log)} | avg latency: {avg_lat:.0f}ms")

llm = LLMClient()
print(f"Backend: {llm.backend}")

<a id='rag'></a>
---
# 🔍 Section 1 — Production RAG (Retrieval-Augmented Generation)
---


## 1.1 What is RAG and Why Does It Matter?

**RAG = Retrieval-Augmented Generation**

LLMs have a fixed knowledge cutoff and can hallucinate. RAG solves this by:
1. **Indexing** your documents into a vector store
2. **Retrieving** the most relevant passages at query time
3. **Augmenting** the LLM prompt with retrieved context

```
User Query
    │
    ▼
  Embed query → vector
    │
    ▼
  Search vector store → top-k relevant chunks
    │
    ▼
  Prompt = system + context + query
    │
    ▼
  LLM generates grounded answer
```

---

## 1.2 RAG Architecture Levels

| Level | Technique | Description |
|---|---|---|
| **Naive** | Dense retrieval | Embed → cosine sim → top-k |
| **Advanced** | Hybrid (BM25 + dense) | Keyword + semantic |
| **Modular** | Reranking, HyDE, MultiQuery | Better retrieval accuracy |
| **Agentic** | RAG-Fusion, Adaptive RAG | Query routing + iteration |

---

## 1.3 Chunking Strategies

| Strategy | Best For | Trade-off |
|---|---|---|
| Fixed size (512 tok) | General use | May cut mid-sentence |
| Sentence | Q&A | Loses cross-sentence context |
| Paragraph | Long-form | Variable size |
| Semantic | Topics | Expensive |
| Hierarchical | Structured docs | Complex retrieval |
| Sliding window | Overlap preserves context | Redundancy |


In [ ]:
# ============================================================
#  PRODUCTION EMBEDDER — Sentence Transformers
# ============================================================

from sentence_transformers import SentenceTransformer
import numpy as np

class ProductionEmbedder:
    """
    Production embedding with:
    - Batch processing for efficiency
    - Caching to avoid recomputation
    - Multiple model support
    - Normalised vectors for cosine similarity
    """

    MODELS = {
        "fast":    "all-MiniLM-L6-v2",      # 22M params, 384-dim, very fast
        "quality": "all-mpnet-base-v2",      # 110M params, 768-dim, best quality
        "multilingual": "paraphrase-multilingual-MiniLM-L12-v2",
    }

    def __init__(self, model_name: str = "fast"):
        self.model_key  = model_name
        self.model_name = self.MODELS.get(model_name, model_name)
        print(f"Loading embedder: {self.model_name}")
        self.model = SentenceTransformer(self.model_name)
        self.dim   = self.model.get_sentence_embedding_dimension()
        self._cache: Dict[str, np.ndarray] = {}
        print(f"✓ Embedder ready  dim={self.dim}")

    def embed(self, texts: Union[str, List[str]],
              batch_size: int = 64,
              normalize: bool = True) -> np.ndarray:
        """Embed texts with caching and batch processing."""
        if isinstance(texts, str):
            texts = [texts]

        results  = np.zeros((len(texts), self.dim), dtype=np.float32)
        to_embed = []
        indices  = []

        for i, text in enumerate(texts):
            key = hashlib.md5(text.encode()).hexdigest()
            if key in self._cache:
                results[i] = self._cache[key]
            else:
                to_embed.append(text)
                indices.append((i, key))

        if to_embed:
            vecs = self.model.encode(
                to_embed, batch_size=batch_size,
                normalize_embeddings=normalize,
                show_progress_bar=False,
                convert_to_numpy=True
            )
            for idx, (arr_i, key) in enumerate(indices):
                self._cache[key] = vecs[idx]
                results[arr_i]   = vecs[idx]

        return results

    def similarity(self, a: np.ndarray, b: np.ndarray) -> float:
        """Cosine similarity (1.0 = identical if normalized)."""
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

    @property
    def cache_size(self) -> int:
        return len(self._cache)


embedder = ProductionEmbedder("fast")
print(f"Dim: {embedder.dim}")

# Test
sample_texts = ["Machine learning is fascinating", "AI and neural networks", "I love pizza"]
vecs = embedder.embed(sample_texts)
print(f"Batch shape: {vecs.shape}")
for i, t1 in enumerate(sample_texts):
    for j, t2 in enumerate(sample_texts[i+1:], i+1):
        sim = embedder.similarity(vecs[i], vecs[j])
        print(f"  sim('{t1[:30]}', '{t2[:30]}') = {sim:.4f}")

In [ ]:
# ============================================================
#  PRODUCTION CHUNKER — Multiple Strategies
# ============================================================

from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class Chunk:
    """A document chunk with metadata for retrieval."""
    id:         str
    text:       str
    doc_id:     str
    doc_title:  str
    chunk_idx:  int
    start_char: int
    end_char:   int
    metadata:   Dict = field(default_factory=dict)
    embedding:  Optional[np.ndarray] = None

    @property
    def token_count(self) -> int:
        return len(self.text.split())

    def __repr__(self):
        return f"Chunk(id={self.id[:8]}, doc={self.doc_title[:20]}, tokens={self.token_count})"


class ProductionChunker:
    """
    Multiple chunking strategies for different document types.
    """

    def __init__(self, chunk_size: int = 256, overlap: int = 32):
        self.chunk_size = chunk_size
        self.overlap    = overlap

    def fixed_size(self, text: str, doc_id: str, doc_title: str) -> List[Chunk]:
        """Fixed-token sliding window chunking."""
        words   = text.split()
        chunks  = []
        step    = self.chunk_size - self.overlap

        for i, start in enumerate(range(0, len(words), step)):
            end      = min(start + self.chunk_size, len(words))
            chunk_words = words[start:end]
            if len(chunk_words) < 20:   # skip tiny final chunk
                break
            chunk_text  = " ".join(chunk_words)
            # Approximate char positions
            char_start  = len(" ".join(words[:start]))
            char_end    = char_start + len(chunk_text)
            chunks.append(Chunk(
                id         = f"{doc_id}_chunk_{i}",
                text       = chunk_text,
                doc_id     = doc_id,
                doc_title  = doc_title,
                chunk_idx  = i,
                start_char = char_start,
                end_char   = char_end,
            ))
            if end == len(words): break

        return chunks

    def sentence_based(self, text: str, doc_id: str, doc_title: str) -> List[Chunk]:
        """Group sentences into chunks respecting sentence boundaries."""
        import re
        sentences = re.split(r'(?<=[.!?])\s+', text)
        chunks, buf, buf_words, idx = [], [], 0, 0

        for sent in sentences:
            words = sent.split()
            if buf_words + len(words) > self.chunk_size and buf:
                chunk_text = " ".join(buf)
                chunks.append(Chunk(
                    id=f"{doc_id}_sent_{idx}", text=chunk_text,
                    doc_id=doc_id, doc_title=doc_title,
                    chunk_idx=idx, start_char=0, end_char=len(chunk_text)))
                # Keep overlap sentences
                overlap_buf = buf[-2:] if len(buf) > 2 else buf
                buf = overlap_buf + [sent]
                buf_words = sum(len(s.split()) for s in buf)
                idx += 1
            else:
                buf.append(sent)
                buf_words += len(words)

        if buf:
            chunk_text = " ".join(buf)
            chunks.append(Chunk(id=f"{doc_id}_sent_{idx}", text=chunk_text,
                                 doc_id=doc_id, doc_title=doc_title,
                                 chunk_idx=idx, start_char=0, end_char=len(chunk_text)))
        return chunks

    def hierarchical(self, text: str, doc_id: str, doc_title: str) -> List[Chunk]:
        """
        Hierarchical: extract section-level and paragraph-level chunks.
        Returns both — retrieval can use either level.
        """
        import re
        sections = re.split(r'\n#{1,3}\s+', text)
        chunks   = []
        for sec_i, section in enumerate(sections):
            # Section-level chunk (for broad context)
            sec_words = section.split()[:self.chunk_size*2]
            chunks.append(Chunk(
                id=f"{doc_id}_sec_{sec_i}", text=" ".join(sec_words),
                doc_id=doc_id, doc_title=doc_title, chunk_idx=sec_i*100,
                start_char=0, end_char=len(" ".join(sec_words)),
                metadata={"level": "section"}))
            # Paragraph-level chunks within section
            paras = [p.strip() for p in section.split("\n\n") if p.strip()]
            for para_i, para in enumerate(paras):
                if len(para.split()) < 10: continue
                chunks.append(Chunk(
                    id=f"{doc_id}_sec{sec_i}_para_{para_i}", text=para,
                    doc_id=doc_id, doc_title=doc_title, chunk_idx=sec_i*100+para_i,
                    start_char=0, end_char=len(para),
                    metadata={"level": "paragraph", "section": sec_i}))
        return chunks


# ── Demo ──────────────────────────────────────────────────────
chunker = ProductionChunker(chunk_size=128, overlap=16)

sample_doc = """
Machine learning is a branch of artificial intelligence that enables systems to learn
from data. Instead of being explicitly programmed, these systems learn patterns and make
decisions with minimal human intervention. The field has seen remarkable growth.

Neural networks are computational models inspired by biological neural networks.
They consist of layers of interconnected nodes that process information. Deep learning
uses many layers to learn complex representations.

Large Language Models like GPT and Claude represent the state of the art in natural
language processing. They are trained on vast amounts of text data using self-supervised
learning objectives.
"""

for strategy_name in ["fixed_size", "sentence_based"]:
    fn = getattr(chunker, strategy_name)
    chunks = fn(sample_doc, "doc_001", "ML Overview")
    print(f"\n{strategy_name}: {len(chunks)} chunks")
    for c in chunks:
        print(f"  {c}  text='{c.text[:50]}...'")

In [ ]:
# ============================================================
#  PRODUCTION VECTOR STORE — FAISS + BM25 Hybrid
# ============================================================

import faiss
from rank_bm25 import BM25Okapi

class ProductionVectorStore:
    """
    Production-grade vector store with:
    - FAISS IndexFlatIP (inner product = cosine for normalized vecs)
    - BM25 sparse index for keyword retrieval
    - Hybrid fusion with Reciprocal Rank Fusion (RRF)
    - Metadata filtering
    - Persistent index (save/load)
    - Thread-safe operations
    """

    def __init__(self, embedder: ProductionEmbedder, use_gpu: bool = False):
        self.embedder  = embedder
        self.dim       = embedder.dim
        self.chunks:   List[Chunk] = []
        self.bm25:     Optional[BM25Okapi] = None
        self._lock     = threading.Lock()

        # FAISS index — FlatIP for exact search (swap for IVF on large corpora)
        self.faiss_idx = faiss.IndexFlatIP(self.dim)
        if use_gpu and faiss.get_num_gpus() > 0:
            res = faiss.StandardGpuResources()
            self.faiss_idx = faiss.index_cpu_to_gpu(res, 0, self.faiss_idx)

    def add(self, chunks: List[Chunk]) -> None:
        """Add chunks to both dense and sparse indices."""
        with self._lock:
            texts = [c.text for c in chunks]
            vecs  = self.embedder.embed(texts, normalize=True)

            for i, (chunk, vec) in enumerate(zip(chunks, vecs)):
                chunk.embedding = vec
                self.chunks.append(chunk)

            self.faiss_idx.add(vecs.astype(np.float32))

            # Rebuild BM25 (could optimise with incremental updates for large corpora)
            tokenized = [c.text.lower().split() for c in self.chunks]
            self.bm25  = BM25Okapi(tokenized)

        print(f"✓ Added {len(chunks)} chunks | Total: {len(self.chunks)}")

    def dense_search(self, query: str, k: int = 10) -> List[Tuple[Chunk, float]]:
        """Semantic search using FAISS."""
        q_vec = self.embedder.embed(query, normalize=True).astype(np.float32)
        scores, indices = self.faiss_idx.search(q_vec, min(k, len(self.chunks)))
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx >= 0:
                results.append((self.chunks[idx], float(score)))
        return results

    def sparse_search(self, query: str, k: int = 10) -> List[Tuple[Chunk, float]]:
        """BM25 keyword search."""
        if not self.bm25: return []
        tokens = query.lower().split()
        scores = self.bm25.get_scores(tokens)
        top_k  = np.argsort(scores)[::-1][:k]
        return [(self.chunks[i], float(scores[i])) for i in top_k if scores[i] > 0]

    def hybrid_search(self, query: str, k: int = 10,
                       alpha: float = 0.6, rrf_k: int = 60) -> List[Tuple[Chunk, float]]:
        """
        Hybrid retrieval = Dense + Sparse fused with Reciprocal Rank Fusion.

        RRF score(d) = Σ 1 / (rrf_k + rank_i(d))

        alpha: weight of dense results in initial candidate selection
        rrf_k: controls rank smoothing (60 is standard)
        """
        dense  = self.dense_search(query, k=k*2)
        sparse = self.sparse_search(query, k=k*2)

        # Build rank maps
        dense_rank  = {c.id: r+1 for r,(c,_) in enumerate(dense)}
        sparse_rank = {c.id: r+1 for r,(c,_) in enumerate(sparse)}
        all_ids     = set(dense_rank) | set(sparse_rank)

        # Compute RRF scores
        rrf_scores: Dict[str, float] = {}
        chunk_map:  Dict[str, Chunk] = {c.id: c for c,_ in dense + sparse}

        for cid in all_ids:
            d_score = 1 / (rrf_k + dense_rank.get(cid, 1e6))
            s_score = 1 / (rrf_k + sparse_rank.get(cid, 1e6))
            rrf_scores[cid] = alpha * d_score + (1-alpha) * s_score

        # Sort and return top-k
        ranked = sorted(rrf_scores.items(), key=lambda x: -x[1])[:k]
        return [(chunk_map[cid], score) for cid, score in ranked]

    def filter_search(self, query: str, k: int = 10,
                       filters: Dict[str, Any] = None) -> List[Tuple[Chunk, float]]:
        """Search with metadata filtering (e.g. filter by doc_id, level)."""
        results = self.hybrid_search(query, k=k*3)
        if not filters: return results[:k]
        filtered = []
        for chunk, score in results:
            match = all(chunk.metadata.get(key) == val for key, val in filters.items())
            if match: filtered.append((chunk, score))
        return filtered[:k]

    @property
    def stats(self) -> Dict:
        return {"total_chunks": len(self.chunks),
                "faiss_vectors": self.faiss_idx.ntotal,
                "embedding_dim": self.dim,
                "has_bm25": self.bm25 is not None}


# ── Build index ───────────────────────────────────────────────
DOCS = [
    ("ML Fundamentals", """
     Machine learning algorithms can be categorized into supervised learning,
     unsupervised learning, and reinforcement learning. Supervised learning
     uses labeled data to train models for classification and regression tasks.
     Common algorithms include linear regression, decision trees, and neural networks.
     Cross-validation and regularization prevent overfitting."""),

    ("Deep Learning", """
     Deep learning uses multi-layer neural networks to learn hierarchical
     representations. Convolutional neural networks excel at image tasks.
     Recurrent neural networks handle sequential data. Transformers use
     self-attention mechanisms and have become the dominant architecture
     for natural language processing tasks."""),

    ("RAG Systems", """
     Retrieval-augmented generation combines vector similarity search with
     large language models. Documents are chunked and embedded into dense
     vector representations. At query time, similar chunks are retrieved
     and provided as context to the LLM. This reduces hallucination and
     allows the model to access current information beyond its training cutoff."""),

    ("Multi-Agent AI", """
     Multi-agent systems use multiple AI models working together to solve
     complex tasks. The ReAct framework combines reasoning and acting.
     Plan-and-execute agents break tasks into subtasks. Self-improving agents
     reflect on their outputs and iteratively improve. Orchestrators coordinate
     multiple specialist agents."""),

    ("LLM Robotics", """
     Large language models can serve as robot brain controllers. Natural language
     commands are parsed into structured robot actions. Safety constraints
     prevent dangerous movements. The Arduino microcontroller interfaces with
     motors, servos, and sensors. Feedback loops allow the robot to adapt
     to its environment using sensor readings."""),
]

chunker = ProductionChunker(chunk_size=80, overlap=10)
store   = ProductionVectorStore(embedder)

all_chunks = []
for title, text in DOCS:
    doc_id = hashlib.md5(title.encode()).hexdigest()[:8]
    chunks = chunker.sentence_based(text.strip(), doc_id, title)
    all_chunks.extend(chunks)

store.add(all_chunks)
print("\nStore stats:", store.stats)

# Test retrieval
query = "How do neural networks learn from data?"
print(f"\nQuery: '{query}'")
print("\nDense results:")
for c, s in store.dense_search(query, k=3):
    print(f"  [{s:.3f}] ({c.doc_title}) {c.text[:80]}...")

print("\nHybrid results (α=0.6):")
for c, s in store.hybrid_search(query, k=3):
    print(f"  [{s:.4f}] ({c.doc_title}) {c.text[:80]}...")

In [ ]:
# ============================================================
#  RERANKER — Cross-Encoder for Precise Relevance Scoring
# ============================================================
#
# Two-stage retrieval (industry standard):
#   Stage 1 — Bi-encoder (fast, approximate): retrieve top-100
#   Stage 2 — Cross-encoder (slow, precise):  rerank to top-10
#
# Cross-encoder sees BOTH query and document simultaneously:
#   score = MLP(BERT([CLS] query [SEP] document [SEP]))
# This is much more accurate but can't be pre-computed.

class CrossEncoderReranker:
    """
    Cross-encoder reranker.
    Falls back to cosine similarity if sentence-transformers cross-encoder unavailable.
    """

    def __init__(self):
        try:
            from sentence_transformers import CrossEncoder
            self.model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2",
                                       max_length=512)
            self.backend = "cross_encoder"
            print("✓ Cross-encoder reranker loaded")
        except Exception:
            self.model   = None
            self.backend = "cosine_fallback"
            print("⚠ Cross-encoder unavailable — using cosine fallback")

    def rerank(self, query: str,
               candidates: List[Tuple[Chunk, float]],
               top_k: int = 5) -> List[Tuple[Chunk, float]]:
        """Rerank candidate chunks by relevance to query."""
        if not candidates: return []

        if self.backend == "cross_encoder":
            pairs  = [(query, c.text) for c,_ in candidates]
            scores = self.model.predict(pairs)
            ranked = sorted(zip(candidates, scores), key=lambda x: -x[1])
            return [(chunk, float(score)) for (chunk,_), score in ranked[:top_k]]
        else:
            # Fallback: re-score by cosine similarity
            q_vec = embedder.embed(query, normalize=True)
            scored = []
            for chunk, _ in candidates:
                if chunk.embedding is None:
                    chunk.embedding = embedder.embed(chunk.text, normalize=True)[0]
                sim = embedder.similarity(q_vec[0], chunk.embedding)
                scored.append((chunk, sim))
            scored.sort(key=lambda x: -x[1])
            return scored[:top_k]


reranker = CrossEncoderReranker()

# End-to-end retrieval with reranking
query = "What is self-attention in transformers?"
print(f"Query: '{query}'")

# Stage 1: fast retrieval
candidates = store.hybrid_search(query, k=10)
print(f"\nStage 1 (hybrid) — {len(candidates)} candidates:")
for c, s in candidates[:4]:
    print(f"  [{s:.4f}] {c.doc_title} | {c.text[:60]}...")

# Stage 2: precise reranking
reranked = reranker.rerank(query, candidates, top_k=4)
print(f"\nStage 2 (reranked) — top {len(reranked)}:")
for c, s in reranked:
    print(f"  [{s:.4f}] {c.doc_title} | {c.text[:60]}...")

In [ ]:
# ============================================================
#  ADVANCED RAG TECHNIQUES
#  1. HyDE  (Hypothetical Document Embeddings)
#  2. Multi-Query Retrieval
#  3. RAG-Fusion
#  4. Self-RAG (with relevance grading)
# ============================================================

class AdvancedRAG:
    """
    Production RAG with advanced retrieval strategies.
    """

    def __init__(self, store: ProductionVectorStore,
                 reranker: CrossEncoderReranker,
                 llm: LLMClient):
        self.store    = store
        self.reranker = reranker
        self.llm      = llm

    # ── 1. HyDE ──────────────────────────────────────────────────
    def hyde_search(self, query: str, k: int = 5) -> List[Tuple[Chunk, float]]:
        """
        Hypothetical Document Embeddings (Gao et al., 2022).

        INSIGHT: Embedding a question and its answer live in different vector spaces.
        Instead, generate a HYPOTHETICAL answer, embed that → much better retrieval.

        query → LLM generates hypothetical answer → embed hypothetical →
        → retrieve real chunks similar to the hypothetical
        """
        hyp_prompt = (
            f"Write a short, factual passage that answers this question:\n"
            f"Question: {query}\n\n"
            f"Write the passage in 2-3 sentences as if from an encyclopedia:"
        )
        hypothetical_doc = self.llm.complete(hyp_prompt, max_tokens=150)
        print(f"  HyDE hypothetical: '{hypothetical_doc[:80]}...'")

        # Embed the hypothetical answer instead of the query
        results = self.store.dense_search(hypothetical_doc, k=k)
        return results

    # ── 2. Multi-Query Retrieval ──────────────────────────────────
    def multi_query_search(self, query: str, n_variants: int = 3,
                            k: int = 5) -> List[Tuple[Chunk, float]]:
        """
        Generate N diverse phrasings of the query, retrieve for each,
        then deduplicate and fuse using RRF.

        Addresses the limitation that a single query phrasing may miss
        relevant documents phrased differently.
        """
        variant_prompt = (
            f"Generate {n_variants} different search queries for this question. "
            f"Each should use different vocabulary but seek the same information.\n"
            f"Question: {query}\n\n"
            f"Return as JSON array of strings: ["query1", "query2", ...]"
        )
        resp = self.llm.complete(variant_prompt, max_tokens=200)

        # Parse variants
        queries = [query]   # always include original
        try:
            m = re.search(r'\[.*?\]', resp, re.DOTALL)
            if m: queries += json.loads(m.group())[:n_variants]
        except Exception:
            queries += [f"What is {query}?", f"Explain {query} in detail"]

        print(f"  Queries ({len(queries)}): {queries}")

        # Retrieve for each query, fuse with RRF
        all_results: Dict[str, Tuple[Chunk, List[float]]] = {}
        for qi, q in enumerate(queries):
            for rank, (chunk, score) in enumerate(self.store.hybrid_search(q, k=k*2)):
                if chunk.id not in all_results:
                    all_results[chunk.id] = (chunk, [])
                rrf_score = 1 / (60 + rank + 1)
                all_results[chunk.id][1].append(rrf_score)

        # Aggregate
        fused = [(chunk, sum(scores)) for chunk, scores in all_results.values()]
        fused.sort(key=lambda x: -x[1])
        return fused[:k]

    # ── 3. Self-RAG ───────────────────────────────────────────────
    def self_rag(self, query: str, k: int = 5) -> Dict[str, Any]:
        """
        Self-RAG (Asai et al., 2023):
        Dynamically decide WHEN to retrieve, WHAT to retrieve, and
        grade retrieved documents for relevance before answering.

        Steps:
        1. Decide if retrieval is needed
        2. Retrieve candidates
        3. Grade each chunk for relevance
        4. Grade generated answer for support
        """
        # Step 1: Is retrieval needed?
        retrieval_prompt = (
            f"Does answering this question require looking up external documents?\n"
            f"Question: {query}\n"
            f"Answer with just YES or NO:"
        )
        needs_retrieval = "yes" in self.llm.complete(retrieval_prompt, max_tokens=10).lower()
        print(f"  Needs retrieval: {needs_retrieval}")

        if not needs_retrieval:
            answer = self.llm.complete(f"Answer: {query}", max_tokens=300)
            return {"answer": answer, "retrieved": [], "graded": [], "mode": "direct"}

        # Step 2: Retrieve
        candidates = self.store.hybrid_search(query, k=k*2)

        # Step 3: Grade chunks for relevance
        graded = []
        for chunk, score in candidates[:k]:
            grade_prompt = (
                f"Is this passage relevant to answering the question?\n"
                f"Question: {query}\nPassage: {chunk.text}\n"
                f"Answer RELEVANT or NOT_RELEVANT:"
            )
            grade = self.llm.complete(grade_prompt, max_tokens=10)
            is_relevant = "relevant" in grade.lower() and "not" not in grade.lower()
            graded.append({"chunk": chunk, "score": score, "relevant": is_relevant})

        relevant = [g for g in graded if g["relevant"]]
        print(f"  Relevant chunks: {len(relevant)}/{len(graded)}")

        # Step 4: Generate answer from relevant chunks
        context = "\n\n".join([f"[{i}] {g['chunk'].text}"
                                  for i, g in enumerate(relevant[:4])])
        answer_prompt = (
            f"Context:\n{context}\n\n"
            f"Question: {query}\n\n"
            f"Answer based ONLY on the context. Cite sources as [0], [1], etc.:"
        )
        answer = self.llm.complete(answer_prompt, max_tokens=400)

        return {"answer": answer, "retrieved": candidates[:k],
                "graded": graded, "mode": "self_rag",
                "relevant_count": len(relevant)}

    # ── 4. Full RAG Pipeline ──────────────────────────────────────
    def answer(self, query: str, strategy: str = "hybrid",
                k: int = 5) -> Dict[str, Any]:
        """Production entry point — routes to appropriate strategy."""
        t0 = time.time()

        if strategy == "hyde":
            retrieved = self.hyde_search(query, k)
        elif strategy == "multi_query":
            retrieved = self.multi_query_search(query, k=k)
        elif strategy == "self_rag":
            return self.self_rag(query, k)
        else:
            retrieved = self.store.hybrid_search(query, k*2)
            retrieved = self.reranker.rerank(query, retrieved, k)

        # Build context
        context = "\n\n".join([f"[{i}] {c.text}" for i, (c,_) in enumerate(retrieved)])
        prompt  = (
            f"Answer the question using ONLY the context below. "
            f"Cite sources [0], [1], etc. If the context is insufficient, say so.\n\n"
            f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
        )
        answer = self.llm.complete(prompt, max_tokens=500)

        return {"answer": answer, "retrieved": retrieved,
                "strategy": strategy, "latency_ms": round((time.time()-t0)*1000)}


rag = AdvancedRAG(store, reranker, llm)

# Compare strategies
queries = [
    "How do transformers use attention mechanisms?",
    "What is the role of LLMs in robotics?",
]

for query in queries:
    print("=" * 60)
    print(f"Query: {query}")
    for strategy in ["hybrid", "hyde", "multi_query"]:
        print(f"\n--- Strategy: {strategy} ---")
        result = rag.answer(query, strategy=strategy, k=4)
        print(f"Answer: {result['answer'][:200]}...")
        print(f"Latency: {result.get('latency_ms', 0)}ms")

In [ ]:
# ============================================================
#  RAG EVALUATION — RAGAS-style Metrics
# ============================================================
#
# Key RAG metrics:
#  Context Recall    — % of ground truth info in retrieved context
#  Context Precision — % of retrieved context that is relevant
#  Faithfulness      — answer is supported by context (no hallucination)
#  Answer Relevancy  — answer addresses the question

class RAGEvaluator:
    """Production RAG evaluation suite."""

    def __init__(self, llm: LLMClient, embedder: ProductionEmbedder):
        self.llm      = llm
        self.embedder = embedder

    def context_precision(self, query: str,
                           retrieved: List[Chunk], k: int = None) -> float:
        """
        What fraction of retrieved chunks are actually relevant?
        Score per chunk: 0 or 1 (LLM-judged)
        """
        chunks = retrieved[:k] if k else retrieved
        if not chunks: return 0.0
        relevant = 0
        for chunk in chunks:
            prompt = (f"Is this passage useful for answering: '{query}'?\n"
                      f"Passage: {chunk.text}\nAnswer YES or NO:")
            ans = self.llm.complete(prompt, max_tokens=5)
            if "yes" in ans.lower(): relevant += 1
        return relevant / len(chunks)

    def faithfulness(self, answer: str, context: List[Chunk]) -> float:
        """
        Are all claims in the answer supported by the context?
        (Anti-hallucination metric)
        """
        ctx_text = "\n".join([c.text for c in context[:4]])
        prompt   = (
            f"Context:\n{ctx_text}\n\n"
            f"Answer: {answer}\n\n"
            f"Rate how well the answer is supported by ONLY the context (0.0-1.0). "
            f"Return just a number:"
        )
        resp = self.llm.complete(prompt, max_tokens=10)
        try:
            return float(re.search(r'\d+\.?\d*', resp).group())
        except: return 0.5

    def answer_relevancy(self, query: str, answer: str) -> float:
        """Does the answer actually address the question?"""
        prompt = (f"Question: {query}\nAnswer: {answer}\n\n"
                  f"Rate how relevant the answer is to the question (0.0-1.0):")
        resp = self.llm.complete(prompt, max_tokens=10)
        try:
            return min(1.0, float(re.search(r'\d+\.?\d*', resp).group()))
        except: return 0.5

    def evaluate_pipeline(self, test_set: List[Dict]) -> pd.DataFrame:
        """Run evaluation over a test set of (query, ground_truth) pairs."""
        records = []
        for item in test_set:
            query  = item["query"]
            result = rag.answer(query, strategy=item.get("strategy","hybrid"), k=5)
            retrieved_chunks = [c for c,_ in result["retrieved"]]
            answer = result["answer"]

            cp  = self.context_precision(query, retrieved_chunks)
            ff  = self.faithfulness(answer, retrieved_chunks)
            ar  = self.answer_relevancy(query, answer)

            records.append({
                "query":     query[:50]+"...",
                "strategy":  item.get("strategy","hybrid"),
                "ctx_prec":  round(cp,3),
                "faithful":  round(ff,3),
                "ans_rel":   round(ar,3),
                "avg_score": round((cp+ff+ar)/3, 3),
                "latency_ms": result.get("latency_ms",0),
            })
        return pd.DataFrame(records)


evaluator = RAGEvaluator(llm, embedder)

test_set = [
    {"query": "What are the components of a RAG system?",    "strategy": "hybrid"},
    {"query": "How does self-attention work in GPT models?", "strategy": "hyde"},
    {"query": "What is reinforcement learning?",             "strategy": "multi_query"},
]

print("Running RAG evaluation...")
df_eval = evaluator.evaluate_pipeline(test_set)
print("\nEvaluation Results:")
print(df_eval.to_string(index=False))

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
metrics = ["ctx_prec", "faithful", "ans_rel"]
x       = np.arange(len(df_eval))
width   = 0.25
colors  = ["#3498db", "#2ecc71", "#e74c3c"]
for i, (m, c) in enumerate(zip(metrics, colors)):
    axes[0].bar(x + i*width, df_eval[m], width, label=m, color=c, edgecolor="white")
axes[0].set_xticks(x + width)
axes[0].set_xticklabels([f"Q{i+1}" for i in range(len(df_eval))])
axes[0].set_ylabel("Score"); axes[0].set_ylim(0,1.1)
axes[0].set_title("RAG Metrics per Query", fontweight="bold")
axes[0].legend()
axes[0].axhline(0.8, color="gray", ls="--", lw=1, label="Production threshold")

axes[1].bar(df_eval["query"].str[:20], df_eval["avg_score"], color="steelblue", edgecolor="white")
axes[1].set_ylabel("Average Score"); axes[1].set_ylim(0,1)
axes[1].set_title("Overall RAG Score per Query", fontweight="bold")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

<a id='mcp'></a>
---
# 🔌 Section 2 — MCP (Model Context Protocol)
---


## 2.1 What is MCP?

**Model Context Protocol** (Anthropic, 2024) is an open standard for connecting LLMs to external tools, data sources, and systems.

Think of it as **USB-C for AI** — a universal interface so any LLM can connect to any tool.

```
┌─────────────────────────────────────────────┐
│              MCP Architecture               │
│                                             │
│  Host (Claude Desktop / your app)           │
│    │                                        │
│    ├── MCP Client ──── MCP Server           │
│    │                     ├── Tools          │
│    │                     ├── Resources      │
│    │                     └── Prompts        │
│    │                                        │
│    └── Multiple Servers simultaneously      │
└─────────────────────────────────────────────┘
```

## 2.2 MCP Primitives

| Primitive | Direction | Description |
|---|---|---|
| **Tools** | LLM calls server | Functions the model can execute |
| **Resources** | Server exposes | Data the model can read (files, DBs) |
| **Prompts** | Server exposes | Reusable prompt templates |
| **Sampling** | Server → LLM | Server requests LLM completions |

## 2.3 Transport Layers

- **stdio** — Local processes (most common for local tools)
- **HTTP + SSE** — Remote servers, web services
- **WebSocket** — Real-time bidirectional communication


In [ ]:
# ============================================================
#  MCP CORE — Protocol Primitives & JSON-RPC Layer
# ============================================================

from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional
import asyncio, json, inspect

# ── Protocol types ────────────────────────────────────────────
@dataclass
class MCPTool:
    """Represents a callable tool exposed by an MCP server."""
    name:        str
    description: str
    parameters:  Dict[str, Any]       # JSON Schema
    handler:     Callable
    tags:        List[str] = field(default_factory=list)

    def to_schema(self) -> Dict:
        return {"name": self.name, "description": self.description,
                "inputSchema": {"type": "object", "properties": self.parameters,
                                "required": list(self.parameters.keys())}}


@dataclass
class MCPResource:
    """Represents a data resource accessible via URI."""
    uri:         str
    name:        str
    description: str
    mime_type:   str = "text/plain"
    handler:     Optional[Callable] = None

    def to_schema(self) -> Dict:
        return {"uri": self.uri, "name": self.name,
                "description": self.description, "mimeType": self.mime_type}


@dataclass
class MCPPrompt:
    """Reusable prompt template with arguments."""
    name:        str
    description: str
    arguments:   List[Dict[str, str]]
    template:    str

    def render(self, **kwargs) -> str:
        result = self.template
        for k, v in kwargs.items():
            result = result.replace(f"{{{{{k}}}}}", str(v))
        return result

    def to_schema(self) -> Dict:
        return {"name": self.name, "description": self.description,
                "arguments": self.arguments}


# ── MCP Server base ────────────────────────────────────────────
class MCPServer:
    """
    Production MCP Server implementation.
    Registers tools, resources, prompts and handles JSON-RPC messages.
    """

    def __init__(self, name: str, version: str = "1.0.0"):
        self.name      = name
        self.version   = version
        self._tools:     Dict[str, MCPTool]    = {}
        self._resources: Dict[str, MCPResource] = {}
        self._prompts:   Dict[str, MCPPrompt]   = {}
        self._call_log:  List[Dict]             = []

    # ── Decorators for registering capabilities ────────────────
    def tool(self, description: str, parameters: Dict = None, tags: List[str] = None):
        """Decorator: @server.tool('description', parameters={...})"""
        def decorator(fn: Callable):
            # Auto-infer parameters from type hints if not provided
            hints  = fn.__annotations__
            params = parameters or {
                k: {"type": "string", "description": f"The {k} parameter"}
                for k in hints if k != "return"
            }
            self._tools[fn.__name__] = MCPTool(
                name=fn.__name__, description=description,
                parameters=params, handler=fn, tags=tags or [])
            return fn
        return decorator

    def resource(self, uri: str, name: str, description: str, mime_type: str = "text/plain"):
        """Decorator: @server.resource('file:///path', 'name', 'desc')"""
        def decorator(fn: Callable):
            self._resources[uri] = MCPResource(
                uri=uri, name=name, description=description,
                mime_type=mime_type, handler=fn)
            return fn
        return decorator

    def prompt(self, name: str, description: str, arguments: List[Dict], template: str):
        """Register a prompt template."""
        self._prompts[name] = MCPPrompt(name, description, arguments, template)

    # ── JSON-RPC message handling ──────────────────────────────
    async def handle_message(self, message: Dict) -> Dict:
        """Handle a JSON-RPC 2.0 message."""
        method = message.get("method")
        params = message.get("params", {})
        id_    = message.get("id")

        handlers = {
            "initialize":          self._handle_initialize,
            "tools/list":          self._handle_tools_list,
            "tools/call":          self._handle_tool_call,
            "resources/list":      self._handle_resources_list,
            "resources/read":      self._handle_resource_read,
            "prompts/list":        self._handle_prompts_list,
            "prompts/get":         self._handle_prompt_get,
        }

        if method not in handlers:
            return self._error(id_, -32601, f"Method not found: {method}")

        try:
            result = await handlers[method](params)
            return {"jsonrpc": "2.0", "id": id_, "result": result}
        except Exception as e:
            return self._error(id_, -32603, str(e))

    async def _handle_initialize(self, params):
        return {"protocolVersion": "2024-11-05",
                "serverInfo": {"name": self.name, "version": self.version},
                "capabilities": {"tools": {}, "resources": {}, "prompts": {}}}

    async def _handle_tools_list(self, params):
        return {"tools": [t.to_schema() for t in self._tools.values()]}

    async def _handle_tool_call(self, params):
        name   = params["name"]
        args   = params.get("arguments", {})
        if name not in self._tools:
            raise ValueError(f"Tool '{name}' not found")

        tool = self._tools[name]
        t0   = time.time()

        # Call handler (sync or async)
        if asyncio.iscoroutinefunction(tool.handler):
            result = await tool.handler(**args)
        else:
            result = tool.handler(**args)

        self._call_log.append({"tool": name, "args": args,
                                "latency_ms": round((time.time()-t0)*1000)})
        return {"content": [{"type": "text", "text": json.dumps(result)
                              if not isinstance(result, str) else result}]}

    async def _handle_resources_list(self, params):
        return {"resources": [r.to_schema() for r in self._resources.values()]}

    async def _handle_resource_read(self, params):
        uri = params["uri"]
        if uri not in self._resources:
            raise ValueError(f"Resource '{uri}' not found")
        content = self._resources[uri].handler()
        return {"contents": [{"uri": uri, "mimeType": self._resources[uri].mime_type,
                               "text": content}]}

    async def _handle_prompts_list(self, params):
        return {"prompts": [p.to_schema() for p in self._prompts.values()]}

    async def _handle_prompt_get(self, params):
        name = params["name"]
        args = params.get("arguments", {})
        if name not in self._prompts:
            raise ValueError(f"Prompt '{name}' not found")
        rendered = self._prompts[name].render(**args)
        return {"prompt": {"name": name, "messages": [{"role":"user","content":rendered}]}}

    def _error(self, id_, code: int, msg: str) -> Dict:
        return {"jsonrpc":"2.0","id":id_,"error":{"code":code,"message":msg}}

    @property
    def registry(self) -> Dict:
        return {"tools": list(self._tools.keys()),
                "resources": list(self._resources.keys()),
                "prompts": list(self._prompts.keys())}


print("✅ MCP Core classes defined")

In [ ]:
# ============================================================
#  PRODUCTION MCP SERVERS
#  1. FileSystem Server  — read/write/search files
#  2. Database Server    — SQL queries
#  3. Web Server         — HTTP requests, search
#  4. Memory Server      — persistent agent memory
# ============================================================

import sqlite3, glob, os

# ══════════════════════════════════════════════════════════════
# SERVER 1: FileSystem MCP Server
# ══════════════════════════════════════════════════════════════
fs_server = MCPServer("filesystem", "1.0.0")

@fs_server.tool("List files in a directory",
                parameters={"path": {"type":"string","description":"Directory path"},
                             "pattern": {"type":"string","description":"Glob pattern","default":"*"}},
                tags=["filesystem"])
def list_files(path: str = ".", pattern: str = "*") -> Dict:
    files = []
    for f in glob.glob(os.path.join(path, pattern)):
        stat = os.stat(f)
        files.append({"name": os.path.basename(f), "size": stat.st_size,
                       "modified": datetime.fromtimestamp(stat.st_mtime).isoformat(),
                       "is_dir": os.path.isdir(f)})
    return {"path": path, "files": files, "count": len(files)}

@fs_server.tool("Read a file",
                parameters={"path": {"type":"string","description":"File path"},
                             "encoding": {"type":"string","description":"Encoding","default":"utf-8"}})
def read_file(path: str, encoding: str = "utf-8") -> str:
    if not os.path.exists(path): return f"Error: File '{path}' not found"
    if os.path.getsize(path) > 1_000_000: return "Error: File too large (>1MB)"
    with open(path, "r", encoding=encoding) as f:
        return f.read()

@fs_server.tool("Write a file",
                parameters={"path":{"type":"string"}, "content":{"type":"string"},
                             "mode":{"type":"string","default":"w"}})
def write_file(path: str, content: str, mode: str = "w") -> Dict:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, mode) as f:
        f.write(content)
    return {"success": True, "path": path, "bytes_written": len(content)}

@fs_server.tool("Search files by content",
                parameters={"directory":{"type":"string"}, "query":{"type":"string"},
                             "extension":{"type":"string","default":".py"}})
def search_files(directory: str, query: str, extension: str = ".py") -> Dict:
    matches = []
    for filepath in glob.glob(f"{directory}/**/*{extension}", recursive=True):
        try:
            with open(filepath, "r", errors="ignore") as f:
                content = f.read()
            if query.lower() in content.lower():
                lines = [f"  L{i+1}: {l.strip()}" for i,l in enumerate(content.split("\n"))
                         if query.lower() in l.lower()]
                matches.append({"file": filepath, "matches": lines[:5]})
        except Exception: pass
    return {"query": query, "matches": matches, "total": len(matches)}

@fs_server.resource("memory://context", "Current Context", "Active working context")
def get_context():
    return json.dumps({"session": str(uuid.uuid4())[:8], "timestamp": datetime.now().isoformat()})

fs_server.prompt("code_review", "Review code for issues",
                  [{"name":"language","description":"Programming language"},
                   {"name":"code","description":"Code to review"}],
                  "Review this {{language}} code for bugs, security issues, and improvements:\n\n```{{language}}\n{{code}}\n```")

# ══════════════════════════════════════════════════════════════
# SERVER 2: Database MCP Server
# ══════════════════════════════════════════════════════════════
db_server = MCPServer("database", "1.0.0")

# Create demo SQLite database
DB_PATH = "/tmp/demo.db"
conn = sqlite3.connect(DB_PATH)
conn.executescript("""
    CREATE TABLE IF NOT EXISTS products (
        id INTEGER PRIMARY KEY, name TEXT, category TEXT,
        price REAL, stock INTEGER, created_at TEXT);
    CREATE TABLE IF NOT EXISTS orders (
        id INTEGER PRIMARY KEY, product_id INTEGER, quantity INTEGER,
        total REAL, status TEXT, created_at TEXT);
""")
# Seed data
import random
categories = ["Electronics", "Books", "Clothing", "Food"]
for i in range(20):
    conn.execute("INSERT OR IGNORE INTO products VALUES (?,?,?,?,?,?)",
                 (i+1, f"Product {i+1}", random.choice(categories),
                  round(random.uniform(10,500),2), random.randint(0,100),
                  datetime.now().isoformat()))
for i in range(50):
    pid = random.randint(1,20)
    qty = random.randint(1,5)
    conn.execute("INSERT OR IGNORE INTO orders VALUES (?,?,?,?,?,?)",
                 (i+1, pid, qty, qty*random.uniform(10,500),
                  random.choice(["pending","shipped","delivered"]),
                  datetime.now().isoformat()))
conn.commit(); conn.close()

@db_server.tool("Execute a SQL query (SELECT only for safety)",
                parameters={"query":{"type":"string","description":"SQL SELECT query"},
                             "limit":{"type":"integer","description":"Max rows","default":100}})
def sql_query(query: str, limit: int = 100) -> Dict:
    # Safety: only allow SELECT
    q = query.strip().upper()
    if not q.startswith("SELECT"):
        return {"error": "Only SELECT queries are allowed for safety"}
    try:
        conn = sqlite3.connect(DB_PATH)
        conn.row_factory = sqlite3.Row
        cur  = conn.execute(query + f" LIMIT {limit}")
        rows = [dict(r) for r in cur.fetchall()]
        cols = [d[0] for d in cur.description] if cur.description else []
        conn.close()
        return {"columns": cols, "rows": rows, "count": len(rows)}
    except Exception as e:
        return {"error": str(e)}

@db_server.tool("List all tables",
                parameters={"schema":{"type":"string","default":"main"}})
def list_tables(schema: str = "main") -> Dict:
    conn = sqlite3.connect(DB_PATH)
    cur  = conn.execute("SELECT name,type FROM sqlite_master WHERE type IN ('table','view')")
    tables = [{"name":r[0],"type":r[1]} for r in cur.fetchall()]
    conn.close()
    return {"tables": tables}

@db_server.tool("Describe a table schema",
                parameters={"table":{"type":"string","description":"Table name"}})
def describe_table(table: str) -> Dict:
    conn = sqlite3.connect(DB_PATH)
    try:
        cur  = conn.execute(f"PRAGMA table_info({table})")
        cols = [{"name":r[1],"type":r[2],"notnull":bool(r[3]),"pk":bool(r[5])}
                for r in cur.fetchall()]
        cnt  = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        conn.close()
        return {"table": table, "columns": cols, "row_count": cnt}
    except Exception as e:
        conn.close(); return {"error": str(e)}

# ══════════════════════════════════════════════════════════════
# SERVER 3: Memory MCP Server (persistent agent memory)
# ══════════════════════════════════════════════════════════════
memory_server = MCPServer("memory", "1.0.0")
_memory_store: Dict[str, Any] = {}

@memory_server.tool("Store information in memory",
                    parameters={"key":{"type":"string"}, "value":{"type":"string"},
                                 "ttl_seconds":{"type":"integer","default":3600}})
def memory_set(key: str, value: str, ttl_seconds: int = 3600) -> Dict:
    _memory_store[key] = {"value": value, "expires": time.time() + ttl_seconds,
                           "created": datetime.now().isoformat()}
    return {"success": True, "key": key}

@memory_server.tool("Retrieve information from memory",
                    parameters={"key":{"type":"string"}})
def memory_get(key: str) -> Dict:
    if key not in _memory_store: return {"found": False, "key": key}
    entry = _memory_store[key]
    if time.time() > entry["expires"]:
        del _memory_store[key]
        return {"found": False, "key": key, "reason": "expired"}
    return {"found": True, "key": key, "value": entry["value"]}

@memory_server.tool("List all memory keys",
                    parameters={"prefix":{"type":"string","default":""}})
def memory_list(prefix: str = "") -> Dict:
    # Prune expired
    now = time.time()
    expired = [k for k,v in _memory_store.items() if now > v["expires"]]
    for k in expired: del _memory_store[k]
    keys = [k for k in _memory_store if k.startswith(prefix)]
    return {"keys": keys, "count": len(keys)}

print("\n=== MCP Servers Registered ===")
for srv in [fs_server, db_server, memory_server]:
    print(f"\n{srv.name} v{srv.version}:")
    for cat, items in srv.registry.items():
        if items: print(f"  {cat}: {items}")

In [ ]:
# ============================================================
#  MCP CLIENT — Routes LLM tool calls to correct server
# ============================================================

class MCPClient:
    """
    Production MCP client that:
    1. Discovers capabilities from multiple servers
    2. Routes tool calls to the right server
    3. Handles errors and retries
    4. Provides tool schemas to the LLM
    """

    def __init__(self, servers: List[MCPServer]):
        self.servers  = {s.name: s for s in servers}
        self._tool_map: Dict[str, str] = {}   # tool_name → server_name
        asyncio.get_event_loop().run_until_complete(self._discover())

    async def _discover(self):
        """Discover all capabilities from connected servers."""
        for srv_name, server in self.servers.items():
            msg    = {"jsonrpc":"2.0","id":1,"method":"tools/list","params":{}}
            result = await server.handle_message(msg)
            for tool in result.get("result",{}).get("tools",[]):
                self._tool_map[tool["name"]] = srv_name

    async def call_tool(self, tool_name: str, args: Dict) -> Any:
        """Route a tool call to the appropriate server."""
        if tool_name not in self._tool_map:
            return {"error": f"Tool '{tool_name}' not found in any server"}
        server = self.servers[self._tool_map[tool_name]]
        msg    = {"jsonrpc":"2.0","id":str(uuid.uuid4()),
                  "method":"tools/call","params":{"name":tool_name,"arguments":args}}
        result = await server.handle_message(msg)
        if "error" in result: return result["error"]
        content = result.get("result",{}).get("content",[{}])
        text    = content[0].get("text","") if content else ""
        try:    return json.loads(text)
        except: return text

    def all_tools_schema(self) -> List[Dict]:
        """Return all tool schemas — fed to the LLM."""
        schemas = []
        for srv in self.servers.values():
            for tool in srv._tools.values():
                s = tool.to_schema()
                s["server"] = srv.name
                schemas.append(s)
        return schemas

    def call_sync(self, tool_name: str, args: Dict) -> Any:
        """Synchronous wrapper for use in non-async contexts."""
        loop = asyncio.new_event_loop()
        try:
            return loop.run_until_complete(self.call_tool(tool_name, args))
        finally:
            loop.close()


# Instantiate client
client = MCPClient([fs_server, db_server, memory_server])

print(f"Registered tools: {list(client._tool_map.keys())}")
print(f"Tool-to-server map: {client._tool_map}")

# Demo tool calls
print("\n=== Tool Call Demos ===")

# DB query
result = client.call_sync("list_tables", {})
print(f"\nTables: {result}")

result = client.call_sync("sql_query", {"query":"SELECT category, COUNT(*) as cnt, AVG(price) as avg_price FROM products GROUP BY category"})
print(f"\nProduct stats:")
for row in result.get("rows", []):
    print(f"  {row}")

# Memory
client.call_sync("memory_set", {"key":"user_preference","value":"dark_mode=true"})
mem = client.call_sync("memory_get", {"key":"user_preference"})
print(f"\nMemory get: {mem}")

# File listing
files = client.call_sync("list_files", {"path":"/tmp","pattern":"*.db"})
print(f"\nDB files: {files}")

In [ ]:
# ============================================================
#  MCP + LLM — Tool-Calling Agent
#  The LLM sees all tool schemas and decides which to call
# ============================================================

class MCPLLMAgent:
    """
    LLM-powered agent using MCP tools.
    Uses a function-calling loop:
    1. Present query + tools to LLM
    2. LLM outputs a tool call JSON
    3. Execute tool via MCP client
    4. Feed result back to LLM
    5. Repeat until LLM gives Final Answer
    """

    def __init__(self, llm: LLMClient, mcp_client: MCPClient, max_steps: int = 10):
        self.llm    = llm
        self.client = mcp_client
        self.max_steps = max_steps

    def run(self, user_request: str, verbose: bool = True) -> Dict:
        tools_schema = self.client.all_tools_schema()
        tools_desc   = "\n".join([
            f"- {t['name']}: {t['description']}  args={list(t['inputSchema']['properties'].keys())}"
            for t in tools_schema
        ])

        system = (
            "You are an AI assistant with access to tools via MCP.\n"
            "Available tools:\n" + tools_desc + "\n\n"
            "To use a tool, respond with JSON:\n"
            '{"tool": "tool_name", "args": {...}}\n'
            "When you have a final answer, respond with:\n"
            '{"final_answer": "your answer here"}'
        )

        history = [f"User: {user_request}"]
        steps   = []
        t0      = time.time()

        for step in range(self.max_steps):
            prompt = "\n".join(history) + "\n\nAssistant:"
            resp   = self.llm.complete(prompt, system=system, max_tokens=500)

            if verbose:
                console.print(f"[bold]Step {step+1}:[/bold] {resp[:150]}")

            # Parse response
            try:
                parsed = json.loads(re.search(r'\{.*?\}', resp, re.DOTALL).group())
            except Exception:
                # Treat as final answer if can't parse
                steps.append({"step":step,"type":"final","content":resp})
                return {"answer": resp, "steps": steps,
                        "latency_ms": round((time.time()-t0)*1000)}

            if "final_answer" in parsed:
                steps.append({"step":step,"type":"final","content":parsed["final_answer"]})
                return {"answer": parsed["final_answer"], "steps": steps,
                        "latency_ms": round((time.time()-t0)*1000)}

            elif "tool" in parsed:
                tool_name = parsed["tool"]
                args      = parsed.get("args", {})
                if verbose:
                    console.print(f"  [yellow]→ Calling tool: {tool_name}({args})[/yellow]")

                tool_result = self.client.call_sync(tool_name, args)
                result_str  = json.dumps(tool_result) if not isinstance(tool_result,str) else tool_result

                history.append(f"Assistant: {resp}")
                history.append(f"Tool({tool_name}): {result_str[:500]}")
                steps.append({"step":step,"type":"tool_call",
                               "tool":tool_name,"args":args,"result":tool_result})
            else:
                break

        return {"answer": "Max steps reached", "steps": steps,
                "latency_ms": round((time.time()-t0)*1000)}


mcp_agent = MCPLLMAgent(llm, client)

print("=== MCP Agent Demo ===\n")
tasks = [
    "What tables are in the database and how many rows does each have?",
    "Store the note 'meeting at 3pm' in memory with key 'today_meeting', then confirm it's stored",
]

for task in tasks:
    print(f"\n{'='*60}")
    print(f"Task: {task}")
    print("─"*60)
    result = mcp_agent.run(task, verbose=True)
    print(f"\nFinal Answer: {result['answer'][:300]}")
    print(f"Steps: {len(result['steps'])} | Latency: {result['latency_ms']}ms")

<a id='agents'></a>
---
# 🤖 Section 3 — Multi-Agent Systems
---


## 3.1 What Are Agents?

An **AI agent** perceives its environment, reasons about it, and takes actions to achieve goals.

```
 Perception → Reasoning → Action → Observation → Reasoning (loop)
```

**Key components:**
- **LLM backbone** — reasoning engine
- **Tools** — ways to act (code execution, web search, APIs)
- **Memory** — short-term (context) + long-term (vector DB)
- **Planning** — decomposing goals into steps

---

## 3.2 Agent Architectures

| Architecture | Paradigm | Best For |
|---|---|---|
| **ReAct** | Reason → Act → Observe (interleaved) | General-purpose, debugging |
| **Plan-and-Execute** | Plan first, then execute | Complex multi-step tasks |
| **Reflection** | Generate → Critique → Refine | Quality-critical outputs |
| **Self-Improving** | Track performance → modify strategy | Adaptive systems |
| **Multi-Agent** | Orchestrator + specialists | Parallelism, scale |

---

## 3.3 ReAct Framework (Yao et al., 2022)

```
Thought:  I need to find the current price of Bitcoin.
Action:   web_search
Input:    "Bitcoin price today"
Observation: Bitcoin is trading at $67,432.

Thought:  I have the price. Now I'll compute 10% of it.
Action:   calculator
Input:    67432 * 0.10
Observation: 6743.2

Thought:  I have all the information needed.
Final Answer: 10% of Bitcoin's current price ($67,432) is $6,743.20
```


In [ ]:
# ============================================================
#  REACT AGENT — Reasoning + Acting + Observing
# ============================================================

from typing import Any, Callable, Dict, List, Optional, Tuple
import re, json, time, math, random

# ── Tool registry ─────────────────────────────────────────────
@dataclass
class Tool:
    name:        str
    description: str
    func:        Callable
    examples:    List[str] = None

    def __call__(self, *args, **kwargs):
        return self.func(*args, **kwargs)


class ToolRegistry:
    def __init__(self):
        self._tools: Dict[str, Tool] = {}

    def register(self, name: str, description: str, examples: List[str] = None):
        def decorator(fn):
            self._tools[name] = Tool(name, description, fn, examples or [])
            return fn
        return decorator

    def __getitem__(self, name): return self._tools[name]
    def __contains__(self, name): return name in self._tools
    def schema(self): return {n: t.description for n, t in self._tools.items()}


tools = ToolRegistry()

@tools.register("calculator", "Evaluate a mathematical expression. Input: math expression string",
                examples=["2+2", "sqrt(16)", "100 * 0.15"])
def calculator(expression: str) -> str:
    try:
        safe = {"sqrt":math.sqrt,"log":math.log,"pi":math.pi,"e":math.e,
                "abs":abs,"round":round,"pow":pow}
        return str(eval(expression, {"__builtins__":{}}, safe))
    except Exception as e: return f"Error: {e}"

@tools.register("web_search", "Search the web for current information. Input: search query",
                examples=["latest AI news", "Python 3.12 features"])
def web_search(query: str) -> str:
    # Mock search results (replace with real search API in production)
    mock_results = {
        "bitcoin": "Bitcoin (BTC) is trading at $67,432, up 2.3% today.",
        "python":  "Python 3.12 released with improved error messages and f-string improvements.",
        "llm":     "Large Language Models like GPT-4 and Claude 3.5 lead 2024 benchmarks.",
        "default": f"Search results for '{query}': Found 3 relevant articles with information about the topic."
    }
    for key, val in mock_results.items():
        if key in query.lower(): return val
    return mock_results["default"]

@tools.register("code_executor", "Execute Python code safely. Input: Python code string",
                examples=["print('hello')", "import math; print(math.pi)"])
def code_executor(code: str) -> str:
    output = []
    safe_globals = {"print": lambda *a: output.append(" ".join(map(str,a))),
                    "math": math, "random": random, "len": len,
                    "range": range, "enumerate": enumerate, "zip": zip,
                    "sum": sum, "max": max, "min": min, "sorted": sorted}
    try:
        exec(code, safe_globals)
        return "
".join(output) if output else "Code executed successfully (no output)"
    except Exception as e:
        return f"Error: {e}"

@tools.register("memory_store", "Store or retrieve information. Usage: 'set key=value' or 'get key'",
                examples=["set result=42", "get result"])
def memory_store(operation: str) -> str:
    _mem = getattr(memory_store, "_store", {})
    memory_store._store = _mem
    if operation.startswith("set "):
        kv = operation[4:].split("=", 1)
        if len(kv) == 2:
            _mem[kv[0].strip()] = kv[1].strip()
            return f"Stored: {kv[0].strip()} = {kv[1].strip()}"
    elif operation.startswith("get "):
        key = operation[4:].strip()
        return _mem.get(key, f"Key '{key}' not found")
    return "Usage: 'set key=value' or 'get key'"

@tools.register("data_analysis", "Analyze data: 'stats [numbers]' or 'sort [numbers]'",
                examples=["stats 1 2 3 4 5", "sort 5 3 1 4 2"])
def data_analysis(command: str) -> str:
    parts = command.split()
    op    = parts[0].lower() if parts else ""
    nums  = []
    for p in parts[1:]:
        try: nums.append(float(p))
        except: pass
    if not nums: return "No numbers provided"
    if op == "stats":
        return (f"count={len(nums)}, mean={sum(nums)/len(nums):.2f}, "
                f"min={min(nums)}, max={max(nums)}, "
                f"std={math.sqrt(sum((x-sum(nums)/len(nums))**2 for x in nums)/len(nums)):.2f}")
    if op == "sort":
        return f"Sorted: {sorted(nums)}"
    return f"Sum: {sum(nums)}, Mean: {sum(nums)/len(nums):.2f}"


# ── ReAct Agent ────────────────────────────────────────────────
class ReActAgent:
    """
    ReAct (Reasoning + Acting) agent.

    Loop:
      1. Thought  — LLM reasons about what to do
      2. Action   — LLM selects a tool
      3. Input    — LLM provides tool arguments
      4. Observation — Tool executes, result fed back
      Repeat until "Final Answer" or max steps.

    KEY DESIGN: The observation is ALWAYS injected back into context.
    The LLM sees its full reasoning history.
    """

    PROMPT_TEMPLATE = """Answer the question using the available tools. Follow this EXACT format:

Thought: [your reasoning about what to do next]
Action: [tool_name]
Action Input: [input to the tool]
Observation: [tool result - filled in for you]
... (repeat as needed)
Thought: I now have enough information.
Final Answer: [your final answer]

Available tools:
{tools}

Question: {question}

{history}""".strip()

    def __init__(self, llm, tools: ToolRegistry, max_steps: int = 8):
        self.llm       = llm
        self.tools     = tools
        self.max_steps = max_steps

    def run(self, question: str, verbose: bool = True) -> Dict:
        history = ""
        steps   = []
        t0      = time.time()

        tools_desc = "
".join([f"- {n}: {d}" for n,d in self.tools.schema().items()])

        if verbose:
            console.print(Panel(f"[bold]Question:[/bold] {question}",
                                 title="ReAct Agent", border_style="blue"))

        for step in range(self.max_steps):
            prompt = self.PROMPT_TEMPLATE.format(
                tools=tools_desc, question=question, history=history)

            response = self.llm.complete(prompt, max_tokens=400)

            # Parse Thought + Action + Action Input
            thought_m  = re.search(r"Thought:\s*(.+?)(?=
Action:|Final Answer:|$)",
                                    response, re.DOTALL)
            action_m   = re.search(r"Action:\s*(\w+)", response)
            input_m    = re.search(r"Action Input:\s*(.+?)(?=
Observation:|$)",
                                    response, re.DOTALL)
            final_m    = re.search(r"Final Answer:\s*(.+?)$", response, re.DOTALL)

            thought  = thought_m.group(1).strip() if thought_m else ""
            action   = action_m.group(1).strip()   if action_m  else ""
            act_input= input_m.group(1).strip()    if input_m   else ""

            if verbose:
                console.print(f"
[bold cyan]Step {step+1}[/bold cyan]")
                if thought: console.print(f"  Thought: {thought}")

            if final_m:
                answer = final_m.group(1).strip()
                steps.append({"step":step,"type":"final","thought":thought,"answer":answer})
                if verbose: console.print(f"
[bold green]Final Answer: {answer}[/bold green]")
                return {"answer": answer, "steps": steps,
                        "latency_ms": round((time.time()-t0)*1000)}

            if action and action in self.tools:
                if verbose:
                    console.print(f"  Action: [yellow]{action}[/yellow]({act_input[:60]})")
                try:
                    observation = str(self.tools[action](act_input))
                except Exception as e:
                    observation = f"Tool error: {e}"

                if verbose:
                    console.print(f"  Observation: [green]{observation[:100]}[/green]")

                history += (f"
Thought: {thought}
Action: {action}
"
                            f"Action Input: {act_input}
Observation: {observation}
")
                steps.append({"step":step,"type":"action","thought":thought,
                               "action":action,"input":act_input,"observation":observation})
            else:
                # No valid action found — use response as-is
                history += f"
{response}
"
                if response.strip():
                    steps.append({"step":step,"type":"direct","content":response})
                else:
                    break

        return {"answer": "Max steps reached", "steps": steps,
                "latency_ms": round((time.time()-t0)*1000)}


# Demo
react = ReActAgent(llm, tools, max_steps=6)

questions = [
    "What is the square root of 144, and then multiply by 7?",
    "Execute code to generate a list of the first 5 perfect squares, then find their mean",
]

for q in questions:
    print("
" + "="*65)
    result = react.run(q, verbose=True)
    print(f"
Completed in {len(result['steps'])} steps | {result['latency_ms']}ms")

In [ ]:
# ============================================================
#  PLAN-AND-EXECUTE AGENT
#  Step 1: Generate a full plan (list of subtasks)
#  Step 2: Execute each subtask with specialized executors
#  Step 3: Aggregate results into final answer
# ============================================================

class PlanAndExecuteAgent:
    """
    Two-phase agent:
    PHASE 1 — PLANNER: breaks complex task into ordered steps
    PHASE 2 — EXECUTOR: executes each step with tools

    Advantages over ReAct:
    - Better for long-horizon tasks
    - Plan can be shown/reviewed before execution
    - Parallel execution of independent steps (if needed)
    - Easier to debug (plan is explicit)
    """

    PLANNER_PROMPT = """You are a task planning AI. Break down the task into clear, ordered subtasks.

Task: {task}

Available tools: {tools}

Return a JSON plan:
{{
  "goal": "overall goal",
  "steps": [
    {{"id": 1, "task": "description", "tool": "tool_name", "input": "...", "depends_on": []}},
    {{"id": 2, "task": "description", "tool": "tool_name", "input": "...", "depends_on": [1]}}
  ]
}}

Rules:
- Use only the listed tools
- Mark dependencies correctly (depends_on = list of step IDs that must complete first)
- Keep each step atomic (one tool call per step)""".strip()

    EXECUTOR_PROMPT = """Execute this step:
Task: {task}
Tool: {tool}
Suggested input: {input}
Previous results: {context}

Use the tool and return just the result.""".strip()

    SYNTHESIZER_PROMPT = """You completed all steps of a plan. Synthesize a comprehensive final answer.

Original task: {task}

Step results:
{results}

Provide a clear, concise final answer:""".strip()

    def __init__(self, llm, tools: ToolRegistry, max_steps: int = 10):
        self.llm       = llm
        self.tools     = tools
        self.max_steps = max_steps

    def plan(self, task: str) -> Dict:
        """Generate execution plan for a task."""
        prompt = self.PLANNER_PROMPT.format(
            task=task,
            tools=json.dumps(self.tools.schema(), indent=2))
        response = self.llm.complete(prompt, max_tokens=600)
        try:
            m = re.search(r'\{.*\}', response, re.DOTALL)
            return json.loads(m.group()) if m else {"goal": task, "steps": []}
        except:
            return {"goal": task, "steps": []}

    def execute_step(self, step: Dict, context: Dict[int, str]) -> str:
        """Execute a single plan step."""
        tool_name = step.get("tool", "")
        task      = step.get("task", "")
        inp       = step.get("input", "")

        # Substitute placeholders from previous results
        for dep_id, dep_result in context.items():
            inp = inp.replace(f"{{{{step_{dep_id}}}}}", str(dep_result))

        if tool_name in self.tools:
            try:
                return str(self.tools[tool_name](inp))
            except Exception as e:
                return f"Tool error: {e}"
        else:
            # LLM handles the step
            prompt = self.EXECUTOR_PROMPT.format(
                task=task, tool=tool_name, input=inp,
                context=json.dumps(context, indent=2))
            return self.llm.complete(prompt, max_tokens=300)

    def run(self, task: str, verbose: bool = True) -> Dict:
        t0 = time.time()

        # Phase 1: Plan
        if verbose: console.print(Panel(f"[bold]Task:[/bold] {task}",
                                         title="Plan-and-Execute Agent", border_style="magenta"))

        plan = self.plan(task)
        steps = plan.get("steps", [])

        if verbose:
            console.print(f"
[bold magenta]Plan ({len(steps)} steps):[/bold magenta]")
            for s in steps:
                deps = f" (after steps {s.get('depends_on',[])})" if s.get('depends_on') else ""
                console.print(f"  Step {s['id']}: [{s.get('tool','?')}] {s['task']}{deps}")

        # Phase 2: Execute
        results: Dict[int, str] = {}
        for step in steps[:self.max_steps]:
            step_id = step["id"]
            deps    = step.get("depends_on", [])

            # Wait for dependencies (in real system: parallel execution where possible)
            dep_context = {d: results[d] for d in deps if d in results}

            if verbose:
                console.print(f"
  [bold]Executing Step {step_id}:[/bold] {step['task']}")

            result = self.execute_step(step, dep_context)
            results[step_id] = result

            if verbose:
                console.print(f"  Result: [green]{result[:100]}[/green]")

        # Phase 3: Synthesize
        results_text = "
".join([f"Step {i}: {r}" for i,r in results.items()])
        synth_prompt = self.SYNTHESIZER_PROMPT.format(task=task, results=results_text)
        final_answer = self.llm.complete(synth_prompt, max_tokens=400)

        if verbose:
            console.print(f"
[bold green]Final Answer: {final_answer[:300]}[/bold green]")

        return {"answer": final_answer, "plan": plan, "step_results": results,
                "latency_ms": round((time.time()-t0)*1000)}


# Demo
planner = PlanAndExecuteAgent(llm, tools)

task = ("Calculate the statistics (mean, std) for the numbers 15, 23, 8, 42, 16, 9, 31. "
        "Then compute what 25% of the mean equals. Store the final result in memory as 'final_calc'.")
result = planner.run(task, verbose=True)

In [ ]:
# ============================================================
#  REFLECTION AGENT — Generate → Critique → Refine
# ============================================================
#
# Inspired by: "Reflexion" (Shinn et al., 2023)
#              "Self-Refine"  (Madaan et al., 2023)
#
# Key insight: LLMs produce better outputs when they critique
# their own work and iteratively refine it — like a human
# proofreading their own essay.

class ReflectionAgent:
    """
    Self-refining agent using a Generate → Critique → Refine loop.

    Can use:
    - Same model for all roles (Reflexion style)
    - Different models for generator vs critic (Debate style)
    """

    GENERATOR_PROMPT = """{task}

Provide a thorough and accurate response:""".strip()

    CRITIC_PROMPT = """Review the following response to this task:

Task: {task}

Response to critique:
{response}

Provide a detailed critique covering:
1. Factual accuracy
2. Completeness (what's missing?)
3. Logic and reasoning errors
4. Clarity and structure
5. Specific improvements needed

Format:
SCORE: [0-10]
ISSUES:
- [issue 1]
- [issue 2]
IMPROVEMENTS:
- [improvement 1]
- [improvement 2]""".strip()

    REFINE_PROMPT = """Improve your previous response based on the critique.

Task: {task}

Your previous response:
{response}

Critique received:
{critique}

Write an improved version that addresses ALL the issues:""".strip()

    def __init__(self, llm, max_iterations: int = 3, score_threshold: float = 7.5):
        self.llm             = llm
        self.max_iterations  = max_iterations
        self.score_threshold = score_threshold

    def _extract_score(self, critique: str) -> float:
        m = re.search(r'SCORE:\s*(\d+\.?\d*)', critique)
        if m:
            return min(10.0, float(m.group(1)))
        # Count positive vs negative signals
        pos = critique.lower().count("good") + critique.lower().count("correct")
        neg = critique.lower().count("issue") + critique.lower().count("missing") + critique.lower().count("error")
        return max(3.0, min(9.0, 5.0 + pos - neg))

    def run(self, task: str, verbose: bool = True) -> Dict:
        t0         = time.time()
        iterations = []

        if verbose:
            console.print(Panel(f"[bold]Task:[/bold] {task}",
                                 title="Reflection Agent", border_style="cyan"))

        # Initial generation
        response = self.llm.complete(
            self.GENERATOR_PROMPT.format(task=task), max_tokens=500)

        if verbose:
            console.print(f"
[bold]Initial Response:[/bold]
{response[:300]}...")

        for i in range(self.max_iterations):
            # Critique
            critique = self.llm.complete(
                self.CRITIC_PROMPT.format(task=task, response=response), max_tokens=400)
            score = self._extract_score(critique)

            if verbose:
                console.print(f"
[bold cyan]Iteration {i+1} Critique (score={score:.1f}/10):[/bold cyan]")
                console.print(f"  {critique[:300]}...")

            iterations.append({"iteration": i+1, "response": response,
                                 "critique": critique, "score": score})

            if score >= self.score_threshold:
                if verbose:
                    console.print(f"
[bold green]✓ Score {score:.1f} ≥ threshold {self.score_threshold}. Done![/bold green]")
                break

            # Refine
            response = self.llm.complete(
                self.REFINE_PROMPT.format(task=task, response=response, critique=critique),
                max_tokens=600)

            if verbose:
                console.print(f"
[bold]Refined Response:[/bold]
{response[:300]}...")

        final_score = iterations[-1]["score"] if iterations else 0
        return {"answer": response, "iterations": iterations,
                "final_score": final_score,
                "num_iterations": len(iterations),
                "latency_ms": round((time.time()-t0)*1000)}


reflection = ReflectionAgent(llm, max_iterations=3, score_threshold=7.0)

tasks = [
    "Explain the difference between supervised and unsupervised learning with examples",
    "Write a Python function to check if a number is prime, with edge case handling",
]

for task in tasks:
    print("
" + "="*65)
    result = reflection.run(task, verbose=True)
    print(f"
Final score: {result['final_score']:.1f}/10 | Iterations: {result['num_iterations']} | {result['latency_ms']}ms")

# Visualise improvement across iterations
if result["iterations"]:
    scores = [it["score"] for it in result["iterations"]]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(range(1, len(scores)+1), scores, "o-", color="steelblue", ms=8, lw=2)
    ax.axhline(7.0, color="red", ls="--", lw=1, label="Threshold (7.0)")
    ax.fill_between(range(1,len(scores)+1), scores, alpha=0.2, color="steelblue")
    ax.set_xlabel("Iteration"); ax.set_ylabel("Quality Score")
    ax.set_title("Reflection Agent — Score Improvement", fontweight="bold")
    ax.set_ylim(0, 10); ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
#  SELF-IMPROVING AGENT
#  Tracks its own performance and modifies its strategy
# ============================================================

class SelfImprovingAgent:
    """
    Agent that learns from its own mistakes over time.

    Mechanisms:
    1. Performance tracking per task type
    2. Failure analysis — what went wrong and why
    3. Strategy adaptation — change approach based on history
    4. Few-shot example curation — keeps best examples
    """

    def __init__(self, llm, tools: ToolRegistry):
        self.llm     = llm
        self.tools   = tools
        self.memory: List[Dict] = []          # All task history
        self.few_shots: List[Dict] = []       # Curated good examples
        self.strategy = "balanced"            # Current strategy
        self.task_stats: Dict[str, List] = {}  # per-type performance

    def _classify_task(self, task: str) -> str:
        """Identify task type for performance tracking."""
        t = task.lower()
        if any(w in t for w in ["calculate","compute","math"]): return "math"
        if any(w in t for w in ["code","program","function","python"]): return "code"
        if any(w in t for w in ["search","find","look up"]): return "search"
        if any(w in t for w in ["analyze","compare","explain"]): return "analysis"
        return "general"

    def _build_strategy_prompt(self) -> str:
        """Build strategy instructions based on past performance."""
        if not self.memory: return ""
        recent  = self.memory[-5:]
        avg_score = sum(m.get("score",5) for m in recent) / len(recent)
        strategies = {
            "conservative": "Be very careful and methodical. Double-check all steps.",
            "balanced":     "Balance thoroughness with efficiency.",
            "aggressive":   "Prioritize speed and use tools liberally.",
        }
        prompt = f"[Strategy: {self.strategy} (avg recent score: {avg_score:.1f}/10)]\n"
        prompt += strategies.get(self.strategy, "")
        if self.few_shots:
            prompt += f"\n\nBest examples from past ({len(self.few_shots)} shots):\n"
            for ex in self.few_shots[-2:]:
                prompt += f"Task: {ex['task'][:50]}... → Score: {ex['score']}/10\n"
        return prompt

    def _adapt_strategy(self):
        """Adapt strategy based on performance history."""
        if len(self.memory) < 3: return
        recent_scores = [m.get("score", 5) for m in self.memory[-5:]]
        avg = sum(recent_scores) / len(recent_scores)
        if avg < 5:
            self.strategy = "conservative"
        elif avg > 7.5:
            self.strategy = "aggressive"
        else:
            self.strategy = "balanced"

    def _self_evaluate(self, task: str, answer: str) -> Dict:
        """Score own response and identify improvements."""
        eval_prompt = (
            f"Evaluate this AI response on a scale 1-10:\n"
            f"Task: {task}\nResponse: {answer}\n\n"
            f"Return JSON: {{\"score\": N, \"strengths\": [...], \"weaknesses\": [...]}}"
        )
        resp = self.llm.complete(eval_prompt, max_tokens=200)
        try:
            m = re.search(r'\{.*\}', resp, re.DOTALL)
            return json.loads(m.group()) if m else {"score": 5, "strengths": [], "weaknesses": []}
        except:
            score_m = re.search(r'\d+', resp)
            return {"score": int(score_m.group()) if score_m else 5,
                    "strengths": [], "weaknesses": []}

    def run(self, task: str, verbose: bool = True) -> Dict:
        t0       = time.time()
        task_type = self._classify_task(task)
        strategy  = self._build_strategy_prompt()

        if verbose:
            console.print(Panel(
                f"[bold]Task:[/bold] {task}\n"
                f"[bold]Type:[/bold] {task_type}  |  [bold]Strategy:[/bold] {self.strategy}",
                title="Self-Improving Agent", border_style="yellow"))

        # Choose execution method based on task type
        if task_type == "math" and "calculator" in self.tools.schema():
            # Try tool first for math
            nums = re.findall(r'[\d.]+', task)
            if nums:
                expr = task.lower()
                for w, op in [("plus","+"),("minus","-"),("times","*"),("divided by","/"),("percent","/100")]:
                    expr = expr.replace(w, op)
                nums_expr = " ".join(nums)
                try:
                    calc_result = str(self.tools["calculator"](nums_expr))
                    answer = f"Using calculator: {nums_expr} = {calc_result}"
                except:
                    answer = self.llm.complete(f"{strategy}\nTask: {task}", max_tokens=400)
            else:
                answer = self.llm.complete(f"{strategy}\nTask: {task}", max_tokens=400)
        else:
            answer = self.llm.complete(f"{strategy}\nTask: {task}", max_tokens=400)

        # Self-evaluate
        evaluation = self._self_evaluate(task, answer)
        score      = evaluation.get("score", 5)

        # Update memory
        record = {"task": task, "answer": answer, "score": score,
                   "task_type": task_type, "strategy": self.strategy,
                   "evaluation": evaluation, "timestamp": time.time()}
        self.memory.append(record)
        if task_type not in self.task_stats:
            self.task_stats[task_type] = []
        self.task_stats[task_type].append(score)

        # Curate few-shots
        if score >= 7:
            self.few_shots.append({"task": task, "answer": answer, "score": score})
            self.few_shots = sorted(self.few_shots, key=lambda x: -x["score"])[:10]

        # Adapt strategy
        self._adapt_strategy()

        if verbose:
            console.print(f"\n[bold]Answer:[/bold] {answer[:300]}")
            console.print(f"\n[bold]Self-Evaluation:[/bold] Score={score}/10")
            console.print(f"  Strengths: {evaluation.get('strengths', [])}")
            console.print(f"  Weaknesses: {evaluation.get('weaknesses', [])}")
            console.print(f"  Strategy adapted to: [yellow]{self.strategy}[/yellow]")

        return {"answer": answer, "score": score, "evaluation": evaluation,
                "latency_ms": round((time.time()-t0)*1000)}

    def performance_report(self):
        """Print performance statistics."""
        if not self.memory:
            print("No history yet")
            return

        console.print(Panel("[bold]Self-Improving Agent Performance Report[/bold]",
                             border_style="yellow"))

        all_scores = [m["score"] for m in self.memory]
        print(f"Total tasks:    {len(self.memory)}")
        print(f"Overall avg:    {sum(all_scores)/len(all_scores):.2f}/10")
        print(f"Curated shots:  {len(self.few_shots)}")
        print(f"Strategy now:   {self.strategy}")
        print(f"\nPer-type stats:")
        for ttype, scores in self.task_stats.items():
            avg = sum(scores)/len(scores)
            print(f"  {ttype:<12}: avg={avg:.1f}, n={len(scores)}")

        # Visualise
        if len(self.memory) > 1:
            scores = [m["score"] for m in self.memory]
            fig, ax = plt.subplots(figsize=(10, 4))
            ax.plot(scores, "o-", color="steelblue", ms=6, lw=2)
            window = min(3, len(scores))
            smoothed = [sum(scores[max(0,i-window):i+1])/len(scores[max(0,i-window):i+1])
                        for i in range(len(scores))]
            ax.plot(smoothed, "r-", lw=2, label="Rolling avg")
            ax.set_xlabel("Task #"); ax.set_ylabel("Score")
            ax.set_title("Self-Improving Agent — Score Over Time", fontweight="bold")
            ax.set_ylim(0,10); ax.legend(); plt.tight_layout(); plt.show()


sia = SelfImprovingAgent(llm, tools)

tasks_sia = [
    "Calculate 15% of 840 and add it to 560",
    "Write a Python function to find all prime numbers up to n",
    "Explain the concept of transfer learning in neural networks",
    "Search for recent developments in large language models",
]

for task in tasks_sia:
    print("\n" + "="*65)
    sia.run(task, verbose=True)

print("\n" + "="*65)
sia.performance_report()

In [ ]:
# ============================================================
#  MULTI-AGENT ORCHESTRATION
#  Orchestrator routes tasks to specialist agents
# ============================================================

class SpecialistAgent:
    """A focused agent for a specific domain."""
    def __init__(self, name: str, domain: str, llm, tools, system_prompt: str = ""):
        self.name          = name
        self.domain        = domain
        self.llm           = llm
        self.tools         = tools
        self.system_prompt = system_prompt or f"You are a specialist in {domain}."
        self._completed    = 0

    def run(self, task: str, context: Dict = None) -> str:
        ctx_str = ""
        if context:
            ctx_str = f"\nContext from other agents:\n{json.dumps(context, indent=2)}\n"
        prompt = f"{self.system_prompt}\n{ctx_str}\nTask: {task}\n\nResponse:"
        result = self.llm.complete(prompt, max_tokens=400)
        self._completed += 1
        return result


class MultiAgentOrchestrator:
    """
    Orchestrator that:
    1. Decomposes the task into subtasks
    2. Routes each subtask to the best specialist
    3. Manages inter-agent communication
    4. Synthesizes final answer

    Architecture:
      User → Orchestrator → [Researcher, Coder, Analyst, Writer] → Orchestrator → User
    """

    def __init__(self, llm, agents: Dict[str, SpecialistAgent]):
        self.llm    = llm
        self.agents = agents

    def decompose(self, task: str) -> List[Dict]:
        """Break task into subtasks, each assigned to an agent."""
        agent_list = "\n".join([f"- {n}: {a.domain}" for n,a in self.agents.items()])
        prompt = (
            f"Break this task into subtasks, each for one specialist:\n"
            f"Task: {task}\nSpecialists:\n{agent_list}\n\n"
            f"Return JSON: [{{\"id\": 1, \"agent\": \"name\", \"subtask\": \"...\", "
            f"\"depends_on\": []}}]"
        )
        resp = self.llm.complete(prompt, max_tokens=400)
        try:
            m = re.search(r'\[.*?\]', resp, re.DOTALL)
            return json.loads(m.group()) if m else []
        except:
            # Fallback: assign whole task to first agent
            first = list(self.agents.keys())[0]
            return [{"id":1,"agent":first,"subtask":task,"depends_on":[]}]

    def run(self, task: str, verbose: bool = True) -> Dict:
        t0 = time.time()
        shared_context: Dict[int, str] = {}

        if verbose:
            console.print(Panel(f"[bold]Task:[/bold] {task}",
                                 title="Multi-Agent Orchestrator", border_style="green"))

        # Decompose
        subtasks = self.decompose(task)
        if verbose:
            console.print(f"\n[bold green]Subtasks ({len(subtasks)}):[/bold green]")
            for s in subtasks:
                console.print(f"  [{s['agent']}] {s['subtask']}")

        # Execute (respecting dependencies)
        results: Dict[int, str] = {}
        for subtask in sorted(subtasks, key=lambda x: len(x.get("depends_on",[]))):
            stid     = subtask["id"]
            agent_nm = subtask["agent"]
            sub_task = subtask["subtask"]
            deps     = subtask.get("depends_on", [])

            # Build context from dependencies
            context = {d: results[d] for d in deps if d in results}
            agent   = self.agents.get(agent_nm, list(self.agents.values())[0])

            if verbose:
                console.print(f"\n  [bold]{agent_nm}[/bold] → {sub_task[:60]}...")

            result = agent.run(sub_task, context)
            results[stid] = result

            if verbose:
                console.print(f"  Result: {result[:120]}...")

        # Synthesize
        results_text = "\n".join([f"Agent {subtasks[i-1]['agent']}: {r}"
                                   for i, r in results.items() if i <= len(subtasks)])
        synth = self.llm.complete(
            f"Synthesize a final answer:\nTask: {task}\n\nAgent results:\n{results_text}\n\nFinal Answer:",
            max_tokens=500)

        if verbose:
            console.print(f"\n[bold green]Final Answer:[/bold green]\n{synth[:400]}")

        # Print agent stats
        if verbose:
            table = Table(title="Agent Performance")
            table.add_column("Agent"); table.add_column("Domain"); table.add_column("Tasks Done")
            for nm, ag in self.agents.items():
                table.add_row(nm, ag.domain, str(ag._completed))
            console.print(table)

        return {"answer": synth, "subtasks": subtasks, "results": results,
                "latency_ms": round((time.time()-t0)*1000)}


# Create specialist agents
specialists = {
    "researcher": SpecialistAgent("researcher", "information gathering and web research", llm, tools,
                                   "You are an expert researcher. Find and summarize information accurately."),
    "coder":      SpecialistAgent("coder", "software development and code analysis", llm, tools,
                                   "You are an expert software engineer. Write clean, efficient code."),
    "analyst":    SpecialistAgent("analyst", "data analysis and statistics", llm, tools,
                                   "You are a data analyst. Provide quantitative insights."),
    "writer":     SpecialistAgent("writer", "content creation and summarization", llm, tools,
                                   "You are a technical writer. Produce clear, concise content."),
}

orchestrator = MultiAgentOrchestrator(llm, specialists)

task = ("Research the top 3 ML frameworks in 2024, analyze their GitHub statistics, "
        "write Python code to visualize a comparison chart, and produce a summary report.")
result = orchestrator.run(task, verbose=True)

<a id='robots'></a>
---
# 🤖 Section 4 — LLM-Controlled Robots & Arduino
---


## 4.1 LLMs as Robot Brains

Large Language Models can serve as high-level robot controllers:

```
Human: "Pick up the red ball and put it in the box"
           │
           ▼
    ┌─────────────────────────────────────────┐
    │    LLM Robot Controller                 │
    │   ┌─────────────────────────────────┐   │
    │   │  Natural Language Understanding │   │
    │   │  Scene Understanding (sensors)  │   │
    │   │  Task Decomposition (planning)  │   │
    │   │  Safety Constraints (rules)     │   │
    │   └─────────────────────────────────┘   │
    └─────────────────────────────────────────┘
           │ Structured Command DSL
           ▼
    ┌─────────────────────────────────────────┐
    │    Low-Level Controller (Arduino)       │
    │   Motor drivers, Servo controllers,     │
    │   Sensor reading, PWM signals           │
    └─────────────────────────────────────────┘
```

## 4.2 Command Domain-Specific Language (DSL)

We define a structured JSON-based robot command language:

```json
{
  "commands": [
    {"action": "move_forward", "params": {"distance_cm": 20, "speed": 150}},
    {"action": "turn_left", "params": {"angle_deg": 90}},
    {"action": "gripper_close", "params": {"force": 80}},
    {"action": "wait", "params": {"ms": 500}}
  ],
  "safety_check": "all_clear",
  "estimated_duration_ms": 2500
}
```

## 4.3 Arduino Serial Protocol

```
Python ──USB/Serial──► Arduino
  │                       │
  │  JSON command          │ Parse JSON
  │  {"action":"move"...}  │ Execute motors
  │◄───────────────────────│ Send sensor data
  │  {"status":"ok",       │
  │   "sensors":{...}}     │
```

## 4.4 Safety Layers

| Layer | Description |
|---|---|
| **L1 - LLM constraint** | System prompt forbids dangerous actions |
| **L2 - DSL validator** | Schema validation before execution |
| **L3 - Safety checker** | Explicit rule engine (no-go zones) |
| **L4 - Hardware limit** | Motor driver current limits |
| **L5 - Emergency stop** | Physical button always overrides software |


In [ ]:
# ============================================================
#  ROBOT COMMAND DSL — Schema, Validator, Parser
# ============================================================

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple
from enum import Enum
import json, re, time, math

class RobotAction(str, Enum):
    """All supported robot actions — exhaustive list for safety."""
    # Locomotion
    MOVE_FORWARD    = "move_forward"
    MOVE_BACKWARD   = "move_backward"
    TURN_LEFT       = "turn_left"
    TURN_RIGHT      = "turn_right"
    STOP            = "stop"
    # Arm / Gripper
    GRIPPER_OPEN    = "gripper_open"
    GRIPPER_CLOSE   = "gripper_close"
    ARM_RAISE       = "arm_raise"
    ARM_LOWER       = "arm_lower"
    ARM_EXTEND      = "arm_extend"
    ARM_RETRACT     = "arm_retract"
    # Head / Sensors
    HEAD_PAN        = "head_pan"
    HEAD_TILT       = "head_tilt"
    CAMERA_CAPTURE  = "camera_capture"
    # Control
    WAIT            = "wait"
    BEEP            = "beep"
    LED_SET         = "led_set"
    EMERGENCY_STOP  = "emergency_stop"


@dataclass
class RobotCommand:
    """Single validated robot command."""
    action:   RobotAction
    params:   Dict[str, Any] = field(default_factory=dict)
    comment:  str = ""

    def to_arduino_bytes(self) -> bytes:
        """Serialize to Arduino serial format: JSON line + newline."""
        msg = {"a": self.action.value, "p": self.params}
        return (json.dumps(msg) + "\n").encode("utf-8")

    def estimated_duration_ms(self) -> int:
        """Estimate how long this command takes to execute."""
        p = self.params
        if self.action == RobotAction.MOVE_FORWARD:
            spd = max(1, p.get("speed", 150))
            return int(p.get("distance_cm", 10) / spd * 1000 * 15)
        elif self.action in (RobotAction.TURN_LEFT, RobotAction.TURN_RIGHT):
            return int(p.get("angle_deg", 90) / 90 * 800)
        elif self.action == RobotAction.WAIT:
            return p.get("ms", 500)
        elif self.action in (RobotAction.GRIPPER_OPEN, RobotAction.GRIPPER_CLOSE):
            return 500
        elif self.action == RobotAction.ARM_RAISE:
            return int(p.get("angle_deg", 45) / 90 * 1000)
        return 200  # default


class RobotCommandSchema:
    """JSON Schema for each robot action's parameters."""

    SCHEMAS: Dict[str, Dict] = {
        "move_forward":  {"distance_cm": (0.1, 500), "speed": (0, 255)},
        "move_backward": {"distance_cm": (0.1, 200), "speed": (0, 200)},  # slower for safety
        "turn_left":     {"angle_deg": (1, 360)},
        "turn_right":    {"angle_deg": (1, 360)},
        "stop":          {},
        "gripper_open":  {"percent": (0, 100)},
        "gripper_close": {"force": (0, 100), "max_width_mm": (0, 150)},
        "arm_raise":     {"angle_deg": (0, 90), "speed": (1, 255)},
        "arm_lower":     {"angle_deg": (0, 90), "speed": (1, 255)},
        "arm_extend":    {"distance_mm": (0, 400)},
        "arm_retract":   {"distance_mm": (0, 400)},
        "head_pan":      {"angle_deg": (-90, 90)},
        "head_tilt":     {"angle_deg": (-30, 60)},
        "camera_capture": {},
        "wait":          {"ms": (0, 10000)},
        "beep":          {"frequency_hz": (100, 5000), "duration_ms": (50, 3000)},
        "led_set":       {"r": (0, 255), "g": (0, 255), "b": (0, 255)},
        "emergency_stop": {},
    }

    @classmethod
    def validate(cls, action: str, params: Dict) -> Tuple[bool, str]:
        """Validate action + params against schema."""
        if action not in cls.SCHEMAS:
            return False, f"Unknown action: '{action}'"
        schema = cls.SCHEMAS[action]
        for param_name, (lo, hi) in schema.items():
            val = params.get(param_name)
            if val is not None:
                if not isinstance(val, (int, float)):
                    return False, f"Param '{param_name}' must be numeric, got {type(val)}"
                if not (lo <= val <= hi):
                    return False, f"Param '{param_name}'={val} out of range [{lo}, {hi}]"
        return True, "OK"


class SafetyChecker:
    """
    Rule-based safety layer — runs BEFORE sending commands to hardware.
    Even if LLM generates valid DSL, safety rules can block execution.
    """

    def __init__(self):
        self.no_go_zones: List[Dict] = []   # [{x1,y1,x2,y2}] in cm
        self.max_speed = 200                 # global speed limit
        self.battery_level = 100.0          # %
        self.emergency_stop_active = False
        self.collision_detected    = False

    def check_command(self, cmd: RobotCommand) -> Tuple[bool, str]:
        """Returns (allowed: bool, reason: str)."""
        if self.emergency_stop_active:
            return False, "Emergency stop is active"
        if self.collision_detected and cmd.action not in (
                RobotAction.STOP, RobotAction.EMERGENCY_STOP,
                RobotAction.MOVE_BACKWARD):
            return False, "Collision detected — only STOP or MOVE_BACKWARD allowed"
        if self.battery_level < 10 and cmd.action not in (
                RobotAction.STOP, RobotAction.EMERGENCY_STOP):
            return False, f"Battery critical ({self.battery_level:.0f}%)"
        speed = cmd.params.get("speed", 0)
        if speed > self.max_speed:
            return False, f"Speed {speed} exceeds max {self.max_speed}"
        dist = cmd.params.get("distance_cm", 0)
        if cmd.action == RobotAction.MOVE_FORWARD and dist > 100:
            return False, f"Single move distance {dist}cm > safety limit 100cm"
        return True, "OK"

    def check_sequence(self, commands: List[RobotCommand]) -> List[Tuple[bool, str]]:
        return [self.check_command(c) for c in commands]


class RobotCommandParser:
    """Parses LLM output into validated RobotCommand objects."""

    def parse_llm_output(self, llm_output: str) -> Tuple[List[RobotCommand], List[str]]:
        """
        Parse LLM JSON output into validated commands.
        Returns: (valid_commands, errors)
        """
        commands = []
        errors   = []

        try:
            m    = re.search(r'\{.*\}', llm_output, re.DOTALL)
            data = json.loads(m.group()) if m else {}
            raw_cmds = data.get("commands", [])
        except Exception as e:
            errors.append(f"JSON parse error: {e}")
            return [], errors

        for i, raw in enumerate(raw_cmds):
            action  = raw.get("action", "")
            params  = raw.get("params", {})
            comment = raw.get("comment", "")

            # Validate action exists
            try:
                robot_action = RobotAction(action)
            except ValueError:
                errors.append(f"Command {i}: unknown action '{action}'")
                continue

            # Validate params against schema
            valid, msg = RobotCommandSchema.validate(action, params)
            if not valid:
                errors.append(f"Command {i} ({action}): {msg}")
                continue

            commands.append(RobotCommand(action=robot_action, params=params, comment=comment))

        return commands, errors


# Demo
parser  = RobotCommandParser()
checker = SafetyChecker()

sample_llm_output = json.dumps({
    "commands": [
        {"action": "move_forward", "params": {"distance_cm": 30, "speed": 150}, "comment": "approach target"},
        {"action": "arm_raise",    "params": {"angle_deg": 45, "speed": 100},   "comment": "prepare gripper"},
        {"action": "gripper_close","params": {"force": 70},                      "comment": "grasp object"},
        {"action": "arm_raise",    "params": {"angle_deg": 30, "speed": 80},    "comment": "lift object"},
        {"action": "turn_right",   "params": {"angle_deg": 90},                  "comment": "face destination"},
        {"action": "move_forward", "params": {"distance_cm": 50, "speed": 120}, "comment": "move to box"},
        {"action": "arm_lower",    "params": {"angle_deg": 30, "speed": 80},    "comment": "lower object"},
        {"action": "gripper_open", "params": {"percent": 100},                   "comment": "release"},
        {"action": "move_backward","params": {"distance_cm": 20, "speed": 100}, "comment": "back away"},
        {"action": "stop",         "params": {},                                  "comment": "done"},
    ],
    "safety_check": "all_clear",
    "estimated_duration_ms": 8500
})

commands, errors = parser.parse_llm_output(sample_llm_output)
safety_checks    = checker.check_sequence(commands)

print(f"Parsed {len(commands)} commands, {len(errors)} errors")
if errors: print(f"Errors: {errors}")

print("\nCommand Sequence:")
total_ms = 0
for cmd, (safe, reason) in zip(commands, safety_checks):
    dur = cmd.estimated_duration_ms()
    total_ms += dur
    status = "✓" if safe else f"✗ ({reason})"
    print(f"  {status} {cmd.action.value:<20} {json.dumps(cmd.params):<40} ~{dur}ms")

print(f"\nTotal estimated duration: {total_ms}ms ({total_ms/1000:.1f}s)")
print(f"Arduino bytes example: {commands[0].to_arduino_bytes()}")

In [ ]:
# ============================================================
#  LLM ROBOT CONTROLLER
#  Natural language → Safety check → Robot commands
# ============================================================

class LLMRobotController:
    """
    Production LLM-based robot controller.

    Pipeline:
    1. Natural language command
    2. Scene context (sensors, camera, map)
    3. LLM generates command DSL (JSON)
    4. Safety validation
    5. Execute on hardware (or simulator)
    """

    SYSTEM_PROMPT = """You are a precision robot controller. Convert natural language commands into safe robot actions.

ROBOT CAPABILITIES:
- Locomotion: move_forward/backward (0-500cm, speed 0-200), turn_left/right (1-360°)
- Arm: arm_raise/lower (0-90°), arm_extend/retract (0-400mm), gripper_open/close
- Sensors: camera_capture, head_pan (-90 to 90°), head_tilt (-30 to 60°)
- Control: wait (0-10000ms), beep, led_set (RGB 0-255), emergency_stop

SAFETY RULES (MANDATORY):
- Maximum single move: 100cm
- Maximum speed: 200
- Always STOP at end of sequence
- Never rotate more than 360° in one command
- If uncertain about safety: output emergency_stop

OUTPUT FORMAT (strict JSON):
{
  "commands": [
    {"action": "action_name", "params": {...}, "comment": "why"}
  ],
  "safety_check": "all_clear" | "warning: <reason>" | "blocked: <reason>",
  "reasoning": "brief explanation of approach",
  "estimated_duration_ms": <number>
}""".strip()

    def __init__(self, llm: LLMClient):
        self.llm      = llm
        self.parser   = RobotCommandParser()
        self.safety   = SafetyChecker()
        self.sim      = RobotSimulator()
        self.history: List[Dict] = []

    def _build_scene_context(self, sensors: Dict) -> str:
        """Format sensor readings for the LLM."""
        parts = []
        if "ultrasonic_cm" in sensors:
            dist = sensors["ultrasonic_cm"]
            parts.append(f"Obstacle ahead: {dist:.1f}cm" if dist < 50
                          else f"Path clear (nearest: {dist:.1f}cm)")
        if "battery_pct" in sensors:
            parts.append(f"Battery: {sensors['battery_pct']:.0f}%")
        if "position" in sensors:
            p = sensors["position"]
            parts.append(f"Position: ({p['x']:.1f}, {p['y']:.1f}) heading={p['heading']:.0f}°")
        if "gripper_state" in sensors:
            parts.append(f"Gripper: {sensors['gripper_state']}")
        return "; ".join(parts) if parts else "No sensor data"

    def execute(self, natural_language: str, sensors: Dict = None,
                 verbose: bool = True) -> Dict:
        """Convert NL command to robot actions and execute."""
        t0 = time.time()
        sensors = sensors or {}

        scene_ctx   = self._build_scene_context(sensors)
        user_prompt = (
            f"Scene: {scene_ctx}\n"
            f"Command: {natural_language}\n\n"
            f"Generate the command sequence:"
        )

        if verbose:
            console.print(Panel(
                f"[bold]Command:[/bold] {natural_language}\n"
                f"[bold]Scene:[/bold] {scene_ctx}",
                title="LLM Robot Controller", border_style="red"))

        # LLM generates commands
        llm_output = self.llm.complete(user_prompt, system=self.SYSTEM_PROMPT, max_tokens=600)

        if verbose:
            console.print(f"\n[dim]LLM Output:[/dim] {llm_output[:200]}...")

        # Parse + validate
        commands, parse_errors = self.parser.parse_llm_output(llm_output)
        safety_results         = self.safety.check_sequence(commands)

        # Filter blocked commands
        safe_commands = [cmd for cmd, (ok, _) in zip(commands, safety_results) if ok]
        blocked       = [(cmd, reason) for cmd, (ok, reason) in zip(commands, safety_results) if not ok]

        if blocked and verbose:
            console.print(f"\n[bold red]Safety blocked {len(blocked)} command(s):[/bold red]")
            for cmd, reason in blocked:
                console.print(f"  ✗ {cmd.action.value}: {reason}")

        # Execute on simulator
        sim_results = []
        if safe_commands:
            if verbose:
                console.print(f"\n[bold green]Executing {len(safe_commands)} commands:[/bold green]")
            for cmd in safe_commands:
                sim_result = self.sim.execute(cmd)
                sim_results.append(sim_result)
                if verbose:
                    icon = "✓" if sim_result["success"] else "✗"
                    console.print(f"  {icon} {cmd.action.value} | {json.dumps(cmd.params)} | {sim_result.get('feedback','')}")

        # Log
        record = {"command": natural_language, "llm_output": llm_output,
                   "parsed_count": len(commands), "safe_count": len(safe_commands),
                   "blocked_count": len(blocked), "sim_results": sim_results,
                   "latency_ms": round((time.time()-t0)*1000)}
        self.history.append(record)

        return record

    def explain_capability(self) -> str:
        """Ask LLM to describe what it can do with this robot."""
        return self.llm.complete(
            "Based on the robot's capabilities (wheels, arm, gripper, sensors), "
            "describe 5 practical tasks this robot could perform in a warehouse.",
            system=self.SYSTEM_PROMPT, max_tokens=400)


class RobotSimulator:
    """
    Software simulator for the robot.
    Tracks state and validates physical feasibility.
    """

    def __init__(self):
        self.x       = 0.0    # cm
        self.y       = 0.0
        self.heading = 0.0    # degrees (0=north)
        self.arm_angle      = 0.0
        self.gripper_state  = "open"
        self.gripper_load   = None
        self.led_color      = (0, 0, 0)
        self.log: List[str] = []

    def execute(self, cmd: RobotCommand) -> Dict:
        """Simulate command execution, update state."""
        a = cmd.action
        p = cmd.params

        if a == RobotAction.MOVE_FORWARD:
            dist = p.get("distance_cm", 10)
            rad  = math.radians(self.heading)
            self.x += dist * math.sin(rad)
            self.y += dist * math.cos(rad)
            feedback = f"Moved to ({self.x:.1f}, {self.y:.1f})"

        elif a == RobotAction.MOVE_BACKWARD:
            dist = p.get("distance_cm", 10)
            rad  = math.radians(self.heading)
            self.x -= dist * math.sin(rad)
            self.y -= dist * math.cos(rad)
            feedback = f"Backed to ({self.x:.1f}, {self.y:.1f})"

        elif a == RobotAction.TURN_LEFT:
            self.heading = (self.heading - p.get("angle_deg", 90)) % 360
            feedback = f"Heading now {self.heading:.0f}°"

        elif a == RobotAction.TURN_RIGHT:
            self.heading = (self.heading + p.get("angle_deg", 90)) % 360
            feedback = f"Heading now {self.heading:.0f}°"

        elif a == RobotAction.ARM_RAISE:
            self.arm_angle = min(90, self.arm_angle + p.get("angle_deg", 30))
            feedback = f"Arm at {self.arm_angle:.0f}°"

        elif a == RobotAction.ARM_LOWER:
            self.arm_angle = max(0, self.arm_angle - p.get("angle_deg", 30))
            feedback = f"Arm at {self.arm_angle:.0f}°"

        elif a == RobotAction.GRIPPER_CLOSE:
            self.gripper_state = "closed"
            feedback = "Gripper closed"

        elif a == RobotAction.GRIPPER_OPEN:
            self.gripper_state = "open"
            self.gripper_load  = None
            feedback = "Gripper opened"

        elif a == RobotAction.LED_SET:
            self.led_color = (p.get("r",0), p.get("g",0), p.get("b",0))
            feedback = f"LED → RGB{self.led_color}"

        elif a == RobotAction.STOP:
            feedback = "Stopped"

        elif a == RobotAction.WAIT:
            feedback = f"Waited {p.get('ms',500)}ms"

        else:
            feedback = f"Executed {a.value}"

        entry = f"{a.value}: {feedback}"
        self.log.append(entry)
        return {"success": True, "feedback": feedback, "state": self.state}

    @property
    def state(self) -> Dict:
        return {"x": round(self.x,2), "y": round(self.y,2), "heading": round(self.heading,1),
                "arm_angle": self.arm_angle, "gripper": self.gripper_state,
                "led": self.led_color}

    def plot_path(self):
        """Visualise robot path from log."""
        # Parse positions from log
        x_vals, y_vals = [0], [0]
        x, y, heading = 0.0, 0.0, 0.0
        for entry in self.log:
            if "Moved to" in entry or "Backed to" in entry:
                m = re.search(r'\(([\d.\-]+),\s*([\d.\-]+)\)', entry)
                if m:
                    x, y = float(m.group(1)), float(m.group(2))
                    x_vals.append(x); y_vals.append(y)

        if len(x_vals) > 1:
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.plot(x_vals, y_vals, "b-o", ms=6, lw=2)
            ax.plot(0, 0, "gs", ms=12, label="Start")
            ax.plot(x_vals[-1], y_vals[-1], "r*", ms=15, label="End")
            for i, (xi, yi) in enumerate(zip(x_vals, y_vals)):
                ax.annotate(str(i), (xi, yi), textcoords="offset points",
                             xytext=(5, 5), fontsize=8)
            ax.set_xlabel("X (cm)"); ax.set_ylabel("Y (cm)")
            ax.set_title("Robot Path Simulation", fontweight="bold")
            ax.legend(); ax.grid(True, alpha=0.3)
            ax.set_aspect("equal")
            plt.tight_layout(); plt.show()


# ── Demo the full pipeline ─────────────────────────────────────
controller = LLMRobotController(llm)

scenarios = [
    {
        "command": "Go forward 40cm, turn right 90 degrees, then go forward 30cm and stop",
        "sensors": {"ultrasonic_cm": 200, "battery_pct": 85,
                     "position": {"x":0,"y":0,"heading":0}, "gripper_state":"open"}
    },
    {
        "command": "Pick up the object in front of me",
        "sensors": {"ultrasonic_cm": 25, "battery_pct": 72,
                     "position": {"x":10,"y":20,"heading":0}, "gripper_state":"open"}
    },
    {
        "command": "Set the LED to blue, beep twice, and go back to origin",
        "sensors": {"battery_pct": 60, "position": {"x":40,"y":30,"heading":45}}
    },
]

for scenario in scenarios:
    print("\n" + "="*65)
    result = controller.execute(scenario["command"], scenario["sensors"], verbose=True)
    print(f"Parsed: {result['parsed_count']} | Safe: {result['safe_count']} | Blocked: {result['blocked_count']}")

# Plot the simulated path
controller.sim.plot_path()

In [ ]:
# ============================================================
#  ARDUINO SERIAL BRIDGE (Full Production Implementation)
# ============================================================
#
# Arduino firmware (C++) counterpart is included in comments.
# Python side uses pyserial for real hardware communication.

import threading, queue, time, json, serial

# ── Arduino Firmware (paste this into Arduino IDE) ────────────
ARDUINO_FIRMWARE = """/*
 * LLM Robot Controller — Arduino Firmware
 * Compatible with: Arduino Uno, Mega, Nano
 *
 * Connections:
 *   Left motor:  IN1=2, IN2=3, ENA=9 (PWM)
 *   Right motor: IN3=4, IN4=7, ENB=10 (PWM)
 *   Servo arm:   pin 11
 *   Servo gripper: pin 12
 *   Ultrasonic:  Trig=5, Echo=6
 *   LED RGB:     R=A0, G=A1, B=A2
 *   Buzzer:      pin 8
 */

#include <Arduino.h>
#include <Servo.h>
#include <ArduinoJson.h>   // Install: Library Manager -> ArduinoJson

// Motor pins
const int IN1=2, IN2=3, ENA=9;
const int IN3=4, IN4=7, ENB=10;

// Servos
Servo armServo, gripperServo;
const int ARM_PIN=11, GRIPPER_PIN=12;
int armAngle=90, gripperPos=90;

// Sensors
const int TRIG=5, ECHO=6;
const int LED_R=A0, LED_G=A1, LED_B=A2;
const int BUZZER=8;

// State
bool emergencyStop = false;

void setup() {
    Serial.begin(115200);
    pinMode(IN1,OUTPUT); pinMode(IN2,OUTPUT); pinMode(ENA,OUTPUT);
    pinMode(IN3,OUTPUT); pinMode(IN4,OUTPUT); pinMode(ENB,OUTPUT);
    armServo.attach(ARM_PIN);
    gripperServo.attach(GRIPPER_PIN);
    pinMode(TRIG,OUTPUT); pinMode(ECHO,INPUT);
    pinMode(LED_R,OUTPUT); pinMode(LED_G,OUTPUT); pinMode(LED_B,OUTPUT);
    pinMode(BUZZER,OUTPUT);
    Serial.println("{\"status\":\"ready\",\"firmware\":\"1.0\"}");
}

float getUltrasonic() {
    digitalWrite(TRIG, LOW); delayMicroseconds(2);
    digitalWrite(TRIG, HIGH); delayMicroseconds(10);
    digitalWrite(TRIG, LOW);
    long dur = pulseIn(ECHO, HIGH, 30000);
    return dur * 0.0343 / 2;
}

void setMotors(int leftSpeed, int rightSpeed) {
    // Left motor
    if (leftSpeed >= 0) { digitalWrite(IN1,HIGH); digitalWrite(IN2,LOW); }
    else { digitalWrite(IN1,LOW); digitalWrite(IN2,HIGH); leftSpeed = -leftSpeed; }
    analogWrite(ENA, constrain(leftSpeed, 0, 255));
    // Right motor
    if (rightSpeed >= 0) { digitalWrite(IN3,HIGH); digitalWrite(IN4,LOW); }
    else { digitalWrite(IN3,LOW); digitalWrite(IN4,HIGH); rightSpeed = -rightSpeed; }
    analogWrite(ENB, constrain(rightSpeed, 0, 255));
}

void sendStatus(const char* action, bool ok, const char* detail="") {
    StaticJsonDocument<256> doc;
    doc["status"]  = ok ? "ok" : "error";
    doc["action"]  = action;
    doc["detail"]  = detail;
    doc["distance"]= getUltrasonic();
    serializeJson(doc, Serial);
    Serial.println();
}

void loop() {
    if (Serial.available()) {
        String line = Serial.readStringUntil('\n');
        StaticJsonDocument<256> cmd;
        DeserializationError err = deserializeJson(cmd, line);
        if (err) { Serial.println("{\"error\":\"parse\"}"); return; }

        if (emergencyStop && strcmp(cmd["a"],"emergency_stop") != 0) {
            Serial.println("{\"error\":\"emergency_stop_active\"}"); return;
        }

        const char* action = cmd["a"];
        JsonObject  params = cmd["p"];

        if (strcmp(action,"move_forward") == 0) {
            int spd = params["speed"] | 150;
            float dist = params["distance_cm"] | 10.0;
            float obs  = getUltrasonic();
            if (obs < 15.0) {
                sendStatus(action, false, "obstacle_detected"); return;
            }
            setMotors(spd, spd);
            delay((long)(dist / spd * 1000 * 15));
            setMotors(0, 0);
            sendStatus(action, true);
        }
        else if (strcmp(action,"move_backward") == 0) {
            int spd = params["speed"] | 100;
            float dist = params["distance_cm"] | 10.0;
            setMotors(-spd, -spd);
            delay((long)(dist / spd * 1000 * 15));
            setMotors(0, 0);
            sendStatus(action, true);
        }
        else if (strcmp(action,"turn_left") == 0) {
            int deg = params["angle_deg"] | 90;
            setMotors(-150, 150);
            delay((long)(deg / 90.0 * 600));
            setMotors(0, 0);
            sendStatus(action, true);
        }
        else if (strcmp(action,"turn_right") == 0) {
            int deg = params["angle_deg"] | 90;
            setMotors(150, -150);
            delay((long)(deg / 90.0 * 600));
            setMotors(0, 0);
            sendStatus(action, true);
        }
        else if (strcmp(action,"arm_raise") == 0) {
            int angle = params["angle_deg"] | 30;
            armAngle = constrain(armAngle + angle, 0, 180);
            armServo.write(armAngle);
            delay(500);
            sendStatus(action, true);
        }
        else if (strcmp(action,"arm_lower") == 0) {
            int angle = params["angle_deg"] | 30;
            armAngle = constrain(armAngle - angle, 0, 180);
            armServo.write(armAngle);
            delay(500);
            sendStatus(action, true);
        }
        else if (strcmp(action,"gripper_close") == 0) {
            gripperServo.write(0);   // close
            delay(600);
            sendStatus(action, true);
        }
        else if (strcmp(action,"gripper_open") == 0) {
            gripperServo.write(90);  // open
            delay(600);
            sendStatus(action, true);
        }
        else if (strcmp(action,"led_set") == 0) {
            analogWrite(LED_R, params["r"] | 0);
            analogWrite(LED_G, params["g"] | 0);
            analogWrite(LED_B, params["b"] | 0);
            sendStatus(action, true);
        }
        else if (strcmp(action,"beep") == 0) {
            int freq = params["frequency_hz"] | 1000;
            int dur  = params["duration_ms"] | 200;
            tone(BUZZER, freq, dur);
            delay(dur + 50);
            sendStatus(action, true);
        }
        else if (strcmp(action,"wait") == 0) {
            delay(params["ms"] | 500);
            sendStatus(action, true);
        }
        else if (strcmp(action,"stop") == 0) {
            setMotors(0, 0);
            sendStatus(action, true);
        }
        else if (strcmp(action,"emergency_stop") == 0) {
            setMotors(0, 0);
            emergencyStop = true;
            Serial.println("{\"status\":\"emergency_stop_activated\"}");
        }
        else {
            sendStatus(action, false, "unknown_action");
        }
    }
}
"""


# ── Python Serial Bridge ──────────────────────────────────────
class ArduinoSerialBridge:
    """
    Production serial bridge for Arduino communication.

    Features:
    - Non-blocking I/O with background reader thread
    - Response queue with timeout
    - Automatic reconnection on disconnect
    - Command rate limiting (prevent buffer overflow)
    - Sensor polling thread
    """

    def __init__(self, port: str = "/dev/ttyUSB0", baud: int = 115200,
                  timeout: float = 2.0, simulate: bool = True):
        self.port     = port
        self.baud     = baud
        self.timeout  = timeout
        self.simulate = simulate
        self.serial_: Optional[serial.Serial] = None
        self._rx_queue: queue.Queue = queue.Queue()
        self._running  = False
        self._sensors  = {}
        self._last_cmd = 0.0
        self._min_cmd_interval = 0.05   # 50ms between commands

        if not simulate:
            self._connect()
        else:
            print(f"⚠ Running in SIMULATION mode (no hardware connected)")
            print(f"  To use real Arduino: set simulate=False, port='{port}'")

    def _connect(self):
        try:
            self.serial_ = serial.Serial(self.port, self.baud, timeout=0.1)
            self._running = True
            self._reader_thread = threading.Thread(target=self._reader, daemon=True)
            self._reader_thread.start()
            print(f"✓ Connected to Arduino on {self.port}")
            time.sleep(2)  # Arduino resets on serial connect
        except serial.SerialException as e:
            print(f"✗ Could not connect to {self.port}: {e}")
            self.simulate = True

    def _reader(self):
        """Background thread — reads serial responses."""
        while self._running and self.serial_:
            try:
                line = self.serial_.readline().decode("utf-8").strip()
                if line:
                    try:
                        data = json.loads(line)
                        self._rx_queue.put(data)
                        if "distance" in data:
                            self._sensors["ultrasonic_cm"] = data["distance"]
                    except json.JSONDecodeError:
                        pass
            except Exception:
                break

    def send_command(self, cmd: RobotCommand) -> Dict:
        """Send command and wait for response (with timeout)."""
        # Rate limiting
        elapsed = time.time() - self._last_cmd
        if elapsed < self._min_cmd_interval:
            time.sleep(self._min_cmd_interval - elapsed)
        self._last_cmd = time.time()

        payload = cmd.to_arduino_bytes()

        if self.simulate:
            return self._simulate_response(cmd)

        if not self.serial_:
            return {"error": "Not connected"}

        self.serial_.write(payload)

        # Wait for response
        deadline = time.time() + self.timeout
        while time.time() < deadline:
            try:
                resp = self._rx_queue.get(timeout=0.1)
                return resp
            except queue.Empty:
                continue
        return {"error": "timeout", "command": cmd.action.value}

    def _simulate_response(self, cmd: RobotCommand) -> Dict:
        """Simulate Arduino response for testing without hardware."""
        dur_ms = cmd.estimated_duration_ms()
        time.sleep(min(dur_ms / 1000, 0.1))   # fast simulation
        return {"status": "ok", "action": cmd.action.value,
                "distance": random.uniform(20, 300), "simulated": True}

    def execute_sequence(self, commands: List[RobotCommand],
                          safety: SafetyChecker, verbose: bool = True) -> List[Dict]:
        """Execute a sequence of commands with safety checking."""
        results = []
        for i, cmd in enumerate(commands):
            ok, reason = safety.check_command(cmd)
            if not ok:
                if verbose: print(f"  ✗ BLOCKED ({reason}): {cmd.action.value}")
                results.append({"blocked": True, "reason": reason})
                if cmd.action != RobotAction.STOP:
                    # Auto-stop on block
                    self.send_command(RobotCommand(RobotAction.STOP, {}))
                break

            resp = self.send_command(cmd)
            results.append(resp)
            if verbose:
                icon = "✓" if resp.get("status") == "ok" else "✗"
                dist = f" [dist={resp['distance']:.0f}cm]" if "distance" in resp else ""
                print(f"  {icon} {cmd.action.value}{dist}")

            # Check for obstacle in response
            if resp.get("detail") == "obstacle_detected":
                print(f"  ⚠ Obstacle detected! Stopping sequence.")
                break

        return results

    def get_sensors(self) -> Dict:
        """Poll all sensors."""
        return dict(self._sensors)

    def close(self):
        self._running = False
        if self.serial_: self.serial_.close()

    def print_firmware(self):
        """Print Arduino firmware for reference."""
        print("\n=== Arduino Firmware (arduino_robot_controller.ino) ===")
        print(ARDUINO_FIRMWARE[:1000] + "\n... [truncated, see full code above]")


# ── Demo ──────────────────────────────────────────────────────
bridge  = ArduinoSerialBridge(simulate=True)
checker = SafetyChecker()

# Full pipeline: NL → LLM → commands → Arduino
controller2 = LLMRobotController(llm)
controller2.sim = bridge  # use bridge as simulator

print("=== Full Production Pipeline Demo ===")
nl_commands = [
    "Do a square: go forward 30cm, turn right, forward 30cm, turn right, forward 30cm, turn right, forward 30cm, stop",
    "Raise the arm 45 degrees, close the gripper, then wait 1 second",
]

for nl in nl_commands:
    print(f"\nCommand: '{nl}'")
    result = controller2.execute(nl, verbose=False)
    commands, _ = controller2.parser.parse_llm_output(result["llm_output"])
    exec_results = bridge.execute_sequence(commands, checker, verbose=True)
    print(f"Executed {len(exec_results)} commands")

bridge.print_firmware()

In [ ]:
# ============================================================
#  CLOSED-LOOP ROBOT CONTROL — Sensor Feedback + LLM Replanning
# ============================================================
#
# The robot continuously:
# 1. Executes a plan
# 2. Reads sensors
# 3. Detects deviations / obstacles
# 4. Replans using LLM if needed

class ClosedLoopRobotController:
    """
    Closed-loop control with sensor feedback and LLM replanning.

    Features:
    - Obstacle avoidance using ultrasonic sensor
    - Goal-tracking with position estimation
    - Automatic replanning on deviation
    - Maximum 3 replan attempts before stopping
    """

    REPLAN_PROMPT = """The robot encountered an issue executing its plan.

Original goal: {goal}
Completed steps: {completed}
Current situation: {situation}
Sensor readings: {sensors}

Generate an alternative plan to achieve the goal from the current position:""".strip()

    def __init__(self, llm: LLMClient, bridge: ArduinoSerialBridge,
                  safety: SafetyChecker):
        self.llm     = llm
        self.bridge  = bridge
        self.safety  = safety
        self.parser  = RobotCommandParser()
        self.sim     = RobotSimulator()
        self.controller = LLMRobotController(llm)

    def _read_sensors(self) -> Dict:
        """Read all sensors and return structured data."""
        hw_sensors = self.bridge.get_sensors()
        return {
            "ultrasonic_cm": hw_sensors.get("ultrasonic_cm", 999),
            "battery_pct":   random.uniform(70, 90),   # simulated
            "position":      {"x": self.sim.x, "y": self.sim.y,
                               "heading": self.sim.heading},
            "arm_angle":     self.sim.arm_angle,
            "gripper_state": self.sim.gripper_state,
            "timestamp":     time.time(),
        }

    def _detect_issue(self, cmd: RobotCommand, response: Dict,
                       sensors: Dict) -> Optional[str]:
        """Detect execution issues from sensor data."""
        if response.get("error"):
            return f"Hardware error: {response['error']}"
        if response.get("detail") == "obstacle_detected":
            return "Obstacle in path"
        if sensors.get("ultrasonic_cm", 999) < 10:
            return "Obstacle too close (<10cm)"
        if sensors.get("battery_pct", 100) < 15:
            return "Critical battery"
        return None

    def execute_goal(self, goal: str, max_replans: int = 3,
                      verbose: bool = True) -> Dict:
        """Execute a high-level goal with sensor feedback and replanning."""
        t0 = time.time()

        if verbose:
            console.print(Panel(f"[bold]Goal:[/bold] {goal}",
                                 title="Closed-Loop Robot Controller", border_style="magenta"))

        completed_steps = []
        replans = 0
        current_goal = goal

        while replans <= max_replans:
            sensors = self._read_sensors()

            # Generate plan
            result = self.controller.execute(current_goal, sensors, verbose=False)
            commands, errors = self.parser.parse_llm_output(result["llm_output"])

            if not commands:
                break

            if verbose:
                console.print(f"\n[bold]Plan ({len(commands)} steps):[/bold]")
                for i, c in enumerate(commands):
                    console.print(f"  {i+1}. {c.action.value} {c.params}")

            # Execute with sensor monitoring
            issue_found = False
            for i, cmd in enumerate(commands):
                sensors = self._read_sensors()

                # Pre-execution safety check with live sensor data
                if (cmd.action in (RobotAction.MOVE_FORWARD,) and
                        sensors["ultrasonic_cm"] < 20):
                    issue_found = True
                    if verbose:
                        console.print(f"  ⚠ Pre-check: Obstacle at {sensors['ultrasonic_cm']:.0f}cm")
                    break

                ok, reason = self.safety.check_command(cmd)
                if not ok:
                    if verbose: console.print(f"  ✗ Safety blocked: {reason}")
                    issue_found = True
                    break

                response = self.bridge.send_command(cmd)
                self.sim.execute(cmd)
                completed_steps.append(cmd.action.value)

                sensors = self._read_sensors()
                issue = self._detect_issue(cmd, response, sensors)

                if verbose:
                    icon = "✓" if not issue else "⚠"
                    console.print(f"  {icon} Step {i+1}: {cmd.action.value} "
                                   f"| dist={sensors['ultrasonic_cm']:.0f}cm")

                if issue:
                    issue_found = True
                    if verbose: console.print(f"  Issue detected: [red]{issue}[/red]")
                    break

            if not issue_found:
                if verbose: console.print("[bold green]\n✓ Goal completed successfully![/bold green]")
                break

            # Replan
            replans += 1
            if replans > max_replans:
                if verbose: console.print(f"[red]Max replans ({max_replans}) reached. Stopping.[/red]")
                self.bridge.send_command(RobotCommand(RobotAction.STOP, {}))
                break

            sensors = self._read_sensors()
            situation = (f"Obstacle at {sensors['ultrasonic_cm']:.0f}cm, "
                          f"at position ({sensors['position']['x']:.0f}, "
                          f"{sensors['position']['y']:.0f})")
            current_goal = (f"Alternative plan for: {goal}. "
                             f"Avoid obstacle {situation}. "
                             f"Already completed: {completed_steps}")

            if verbose:
                console.print(f"\n[yellow]Replanning ({replans}/{max_replans})...[/yellow]")

        return {"goal": goal, "completed_steps": completed_steps,
                "replans": replans, "final_position": self.sim.state,
                "latency_ms": round((time.time()-t0)*1000)}


# Demo closed-loop
cl_controller = ClosedLoopRobotController(llm, bridge, checker)
result = cl_controller.execute_goal(
    "Navigate to position 50cm forward, 30cm right from start, pick up object",
    max_replans=2, verbose=True)

print(f"\nFinal position: {result['final_position']}")
print(f"Completed steps: {result['completed_steps']}")
print(f"Replans needed: {result['replans']}")

# Visualise path
cl_controller.sim.plot_path()

---
# 📋 Production LLM Systems — Reference Card
---

## RAG Cheat Sheet

```
CHUNKING:   fixed_size < sentence < semantic < hierarchical
RETRIEVAL:  sparse (BM25) + dense (FAISS) → hybrid (RRF)
RERANKING:  bi-encoder (fast) → cross-encoder (precise)
ADVANCED:   HyDE | Multi-Query | RAG-Fusion | Self-RAG
EVAL:       context_precision, faithfulness, answer_relevancy
```

## MCP Protocol Flow

```
Client → initialize → Server
Client → tools/list → [tool schemas]
Client → tools/call {name, args} → {content: [{text: result}]}
Client → resources/read {uri} → {contents: [{text}]}
Client → prompts/get {name, args} → {messages: [rendered]}
```

## Agent Pattern Selection

| If you need | Use |
|---|---|
| General-purpose tool use | ReAct |
| Long multi-step tasks | Plan-and-Execute |
| High-quality outputs | Reflection |
| Adaptive learning | Self-Improving |
| Parallelism / specialization | Multi-Agent |
| Uncertainty handling | ReAct + Self-RAG |

## Robot Command Quick Reference

```python
# NL → Commands
controller.execute("Go forward 50cm and stop", sensors)

# Direct commands
cmd = RobotCommand(RobotAction.MOVE_FORWARD, {"distance_cm": 30, "speed": 150})
bridge.send_command(cmd)

# Safety check (always before hardware)
ok, reason = checker.check_command(cmd)

# Closed-loop with replanning
cl_controller.execute_goal("Navigate to position X, pick up object")
```

## Production Checklist

- [ ] API key rotation and rate limiting
- [ ] Vector store persistence (save/load index)
- [ ] Structured logging (every LLM call, latency, tokens)
- [ ] Fallback chain (primary LLM → backup → rule-based)
- [ ] Agent timeout and max-step limits
- [ ] Hardware safety: physical E-stop always overrides software
- [ ] Sensor validation: sanity-check readings before reacting
- [ ] Cost tracking: tokens used per task
- [ ] Human-in-the-loop for high-stakes robot actions
- [ ] Full audit trail for compliance

---
*Production LLM Systems: RAG · MCP · Multi-Agent · Robot Control*
